In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
#ADC_WORKS_VARIANTS_WITH_ROW_NUMBERS

# import re
# import snowflake.snowpark.functions as F
# from snowflake.snowpark.functions import col, lit, regexp_replace, when

# # SQL UDFs for core string manipulation functions
# def register_udfs(session):
#     print("Registering JavaScript UDFs for title variant generation...")
    
#     # Register the JavaScript UDFs
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.IS_ARTICLE(word STRING)
#     RETURNS BOOLEAN
#     LANGUAGE JAVASCRIPT
#     AS $$
#         const articles = ['A', 'AN', 'THE'];
#         return articles.includes(WORD.toUpperCase());
#     $$;
#     """).collect()
    
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.REMOVE_ARTICLES(title STRING)
#     RETURNS STRING
#     LANGUAGE JAVASCRIPT
#     AS $$
#         if (!TITLE) return '';
        
#         const articles = ['A', 'AN', 'THE'];
#         const words = TITLE.trim().split(/\s+/);
        
#         // Remove leading article
#         if (words.length > 0 && articles.includes(words[0].toUpperCase())) {
#             words.shift();
#         }
        
#         // Remove trailing article
#         if (words.length > 0 && articles.includes(words[words.length - 1].toUpperCase())) {
#             words.pop();
#         }
        
#         return words.join(' ');
#     $$;
#     """).collect()
    
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.NORMALIZE_WHITESPACE(title STRING)
#     RETURNS STRING
#     LANGUAGE JAVASCRIPT
#     AS $$
#         if (!TITLE) return '';
#         return TITLE.trim().replace(/\s+/g, ' ');
#     $$;
#     """).collect()
    
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.CLEAN_UNMATCHED_BRACKETS(title STRING)
#     RETURNS STRING
#     LANGUAGE JAVASCRIPT
#     AS $$
#         if (!TITLE) return '';
        
#         const stack = [];
#         const bracketPairs = {'(': ')', '[': ']', '{': '}'};
#         const reversePairs = {')': '(', ']': '[', '}': '{'};
#         const skipIndices = new Set();
        
#         // First pass - identify unmatched brackets
#         for (let i = 0; i < TITLE.length; i++) {
#             const char = TITLE[i];
            
#             if (bracketPairs[char]) {  // Opening bracket
#                 stack.push([char, i]);
#             } else if (reversePairs[char]) {  // Closing bracket
#                 if (stack.length && stack[stack.length - 1][0] === reversePairs[char]) {
#                     stack.pop();  // Matched pair
#                 } else {
#                     // Unmatched closing bracket - mark for removal
#                     skipIndices.add(i);
#                 }
#             }
#         }
        
#         // Any remaining opening brackets are unmatched
#         for (const [_, idx] of stack) {
#             skipIndices.add(idx);
#         }
        
#         // Build cleaned title without unmatched brackets
#         let cleanedTitle = "";
#         for (let i = 0; i < TITLE.length; i++) {
#             if (!skipIndices.has(i)) {
#                 cleanedTitle += TITLE[i];
#             }
#         }
        
#         return cleanedTitle.trim().replace(/\s+/g, ' ');
#     $$;
#     """).collect()
    
#     # Complex variant generation function
#     # This is a simplified JavaScript version that returns JSON with variants
#     session.sql("""
#     CREATE OR REPLACE FUNCTION EDW_APPS.MATCHING.GENERATE_TITLE_VARIANTS_JS(title STRING)
#     RETURNS VARIANT
#     LANGUAGE JAVASCRIPT
#     AS $$
#         // Handle NULL titles
#         if (TITLE === null || TITLE === undefined) {
#             return [];
#         }
        
#         const originalTitle = TITLE.trim();
        
#         // Helper functions
#         function isArticle(word) {
#             const articles = ['A', 'AN', 'THE'];
#             return articles.includes(word.toUpperCase());
#         }
        
#         function removeArticles(title) {
#             if (!title) return '';
#             const words = title.trim().split(/\s+/);
            
#             if (words.length > 0 && isArticle(words[0])) {
#                 words.shift();
#             }
            
#             if (words.length > 0 && isArticle(words[words.length - 1])) {
#                 words.pop();
#             }
            
#             return words.join(' ');
#         }
        
#         function normalizeWhitespace(title) {
#             if (!title) return '';
#             return title.trim().replace(/\s+/g, ' ');
#         }
        
#         function cleanUnmatchedBrackets(title) {
#             if (!title) return '';
            
#             const stack = [];
#             const bracketPairs = {'(': ')', '[': ']', '{': '}'};
#             const reversePairs = {')': '(', ']': '[', '}': '{'};
#             const skipIndices = new Set();
            
#             // First pass - identify unmatched brackets
#             for (let i = 0; i < title.length; i++) {
#                 const char = title[i];
                
#                 if (bracketPairs[char]) {  // Opening bracket
#                     stack.push([char, i]);
#                 } else if (reversePairs[char]) {  // Closing bracket
#                     if (stack.length && stack[stack.length - 1][0] === reversePairs[char]) {
#                         stack.pop();  // Matched pair
#                     } else {
#                         // Unmatched closing bracket - mark for removal
#                         skipIndices.add(i);
#                     }
#                 }
#             }
            
#             // Any remaining opening brackets are unmatched
#             for (const [_, idx] of stack) {
#                 skipIndices.add(idx);
#             }
            
#             // Build cleaned title without unmatched brackets
#             let cleanedTitle = "";
#             for (let i = 0; i < title.length; i++) {
#                 if (!skipIndices.has(i)) {
#                     cleanedTitle += title[i];
#                 }
#             }
            
#             return normalizeWhitespace(cleanedTitle);
#         }
        
#         // Main variant generation logic
#         const cleanedTitle = cleanUnmatchedBrackets(originalTitle);
#         const variants = new Set();
        
#         // FIXED: Store only the regex pattern source, not the regex object itself
#         const bracketPatternSource = /([\[\(\{])([^\[\]\(\)\{\}]*)([\]\)\}])/g.source;
        
#         // Find all matched brackets in the cleaned title
#         const bracketMatches = [];
#         let match;
#         // FIXED: Always create a new RegExp instance when using the pattern
#         const patternForMatching = new RegExp(bracketPatternSource, 'g');
#         while ((match = patternForMatching.exec(cleanedTitle)) !== null) {
#             bracketMatches.push({
#                 full: match[0],
#                 open: match[1],
#                 content: match[2],
#                 close: match[3],
#                 start: match.index,
#                 end: match.index + match[0].length
#             });
#         }
        
#         // If no brackets, check if original had any
#         if (bracketMatches.length === 0) {
#             if (/[\[\]\(\)\{\}]/.test(originalTitle)) {
#                 const cleanTitle = removeArticles(cleanedTitle);
#                 if (cleanTitle) {
#                     variants.add(cleanTitle);
#                 }
#             }
#             return Array.from(variants).sort();
#         }
        
#         // FIXED: Create a new RegExp instance for each replacement
#         const noBrackets = cleanedTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
#         const normalizedNoBrackets = normalizeWhitespace(noBrackets);
#         const noArticlesNoBrackets = removeArticles(normalizedNoBrackets);
        
#         if (noArticlesNoBrackets && 
#             noArticlesNoBrackets.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#             variants.add(noArticlesNoBrackets);
#         }
        
#         // Group adjacent brackets
#         const bracketGroups = [];
#         if (bracketMatches.length > 0) {
#             let currentGroup = [bracketMatches[0]];
            
#             for (let i = 1; i < bracketMatches.length; i++) {
#                 const prevEnd = bracketMatches[i-1].end;
#                 const currentStart = bracketMatches[i].start;
                
#                 // Check if there's text between brackets
#                 if (cleanedTitle.substring(prevEnd, currentStart).trim()) {
#                     // Non-adjacent brackets, start a new group
#                     bracketGroups.push({
#                         start: currentGroup[0].start,
#                         end: currentGroup[currentGroup.length-1].end,
#                         matches: [...currentGroup]
#                     });
#                     currentGroup = [bracketMatches[i]];
#                 } else {
#                     // Adjacent brackets, add to current group
#                     currentGroup.push(bracketMatches[i]);
#                 }
#             }
            
#             // Add the last group
#             bracketGroups.push({
#                 start: currentGroup[0].start,
#                 end: currentGroup[currentGroup.length-1].end,
#                 matches: [...currentGroup]
#             });
#         }
        
#         // Split the title into segments: text segments and bracket groups
#         const segments = [];
#         let lastEnd = 0;
        
#         for (const group of bracketGroups) {
#             // Add text before this group
#             if (group.start > lastEnd) {
#                 segments.push({
#                     type: 'text',
#                     content: cleanedTitle.substring(lastEnd, group.start)
#                 });
#             }
            
#             // Add this bracket group
#             segments.push({
#                 type: 'group',
#                 content: group.matches
#             });
            
#             // Update lastEnd
#             lastEnd = group.end;
#         }
        
#         // Add any remaining text after the last bracket group
#         if (lastEnd < cleanedTitle.length) {
#             segments.push({
#                 type: 'text',
#                 content: cleanedTitle.substring(lastEnd)
#             });
#         }
        
#         // Variant 1: with first bracket of first group only
#         const firstVariantParts = [];
        
#         for (const segment of segments) {
#             if (segment.type === 'text') {
#                 firstVariantParts.push(segment.content);
#             } else if (segment.type === 'group') {
#                 // Add only first bracket content from first group
#                 firstVariantParts.push(segment.content[0].content);
#             }
#         }
        
#         const firstVariant = removeArticles(normalizeWhitespace(firstVariantParts.join(' ')));
        
#         if (firstVariant && 
#             !variants.has(firstVariant) && 
#             firstVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#             variants.add(firstVariant);
#         }
        
#         // Variant 2: If multiple groups, create variant with first bracket of first group
#         // and first bracket of last group
#         if (bracketGroups.length > 1) {
#             const secondVariantParts = [];
#             let firstGroupUsed = false;
#             let lastGroupUsed = false;
            
#             for (const segment of segments) {
#                 if (segment.type === 'text') {
#                     secondVariantParts.push(segment.content);
#                 } else if (segment.type === 'group') {
#                     // Check if this is first group
#                     if (!firstGroupUsed && segment.content[0].start === bracketGroups[0].matches[0].start) {
#                         secondVariantParts.push(segment.content[0].content);
#                         firstGroupUsed = true;
#                     } 
#                     // Check if this is last group
#                     else if (!lastGroupUsed && 
#                             segment.content[0].start === bracketGroups[bracketGroups.length-1].matches[0].start &&
#                             bracketGroups[0].start !== bracketGroups[bracketGroups.length-1].start) {
#                         secondVariantParts.push(segment.content[0].content);
#                         lastGroupUsed = true;
#                     }
#                 }
#             }
            
#             const secondVariant = removeArticles(normalizeWhitespace(secondVariantParts.join(' ')));
            
#             if (secondVariant && 
#                 !variants.has(secondVariant) && 
#                 secondVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                 variants.add(secondVariant);
#             }
#         }
        
#         // Variant 3: If multiple groups, create variant with just text and first bracket of last group
#         if (bracketGroups.length > 1) {
#             const thirdVariantParts = [];
#             let lastGroupUsed = false;
            
#             for (const segment of segments) {
#                 if (segment.type === 'text') {
#                     thirdVariantParts.push(segment.content);
#                 } else if (segment.type === 'group' && !lastGroupUsed && 
#                         segment.content[0].start === bracketGroups[bracketGroups.length-1].matches[0].start) {
#                     thirdVariantParts.push(segment.content[0].content);
#                     lastGroupUsed = true;
#                 }
#             }
            
#             const thirdVariant = removeArticles(normalizeWhitespace(thirdVariantParts.join(' ')));
            
#             if (thirdVariant && 
#                 !variants.has(thirdVariant) && 
#                 thirdVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                 variants.add(thirdVariant);
#             }
#         }
        
#         // Variant 4: Handle titles that start with a bracket
#         if (bracketMatches.length > 0 && bracketMatches[0].start === 0) {
#             // Get just the content of the first bracket
#             const bracketContent = normalizeWhitespace(bracketMatches[0].content);
#             let restOfTitle = cleanedTitle.substring(bracketMatches[0].end);
            
#             // FIXED: Use a new RegExp instance for replacement
#             restOfTitle = restOfTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
#             restOfTitle = normalizeWhitespace(restOfTitle);
            
#             // Create variant with just bracket content + rest of title
#             const combined = removeArticles(normalizeWhitespace(`${bracketContent} ${restOfTitle}`));
            
#             if (combined && 
#                 !variants.has(combined) && 
#                 combined.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                 variants.add(combined);
#             }
            
#             // Also create a variant with just the bracket content if it's followed by more brackets
#             if (bracketMatches.length > 1 && bracketMatches[1].start === bracketMatches[0].end) {
#                 const justContent = removeArticles(bracketContent);
                
#                 if (justContent && 
#                     !variants.has(justContent) && 
#                     justContent.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                     variants.add(justContent);
#                 }
#             }
#         }
        
#         // Complex case handling for titles like "(NEW WORLD) (HI) HELLO (SMILE TIME) WORLD (TIME) (XXXX)"
#         if (bracketGroups.length >= 2) {
#             // Find brackets that aren't in any group (middle brackets)
#             const middleBrackets = [];
            
#             for (const match of bracketMatches) {
#                 let isInGroup = false;
                
#                 for (const group of bracketGroups) {
#                     for (const groupMatch of group.matches) {
#                         if (match.start === groupMatch.start && match.end === groupMatch.end) {
#                             isInGroup = true;
#                             break;
#                         }
#                     }
                    
#                     if (isInGroup) break;
#                 }
                
#                 if (!isInGroup) {
#                     middleBrackets.push(match);
#                 }
#             }
            
#             // If we have middle brackets between groups
#             if (middleBrackets.length > 0) {
#                 // Create variant with first group content + middle brackets + last group content
#                 const complexVariantParts = [];
#                 let baseTextAdded = false;
                
#                 // Add first group content
#                 complexVariantParts.push(bracketGroups[0].matches[0].content);
                
#                 // FIXED: Use a new RegExp instance for replacement
#                 const baseText = cleanedTitle.replace(new RegExp(bracketPatternSource, 'g'), '');
#                 complexVariantParts.push(baseText);
#                 baseTextAdded = true;
                
#                 // Add content from middle brackets
#                 for (const middleMatch of middleBrackets) {
#                     complexVariantParts.push(middleMatch.content);
#                 }
                
#                 // Add last group content (first bracket only)
#                 complexVariantParts.push(bracketGroups[bracketGroups.length-1].matches[0].content);
                
#                 const complexVariant = removeArticles(normalizeWhitespace(complexVariantParts.join(' ')));
                
#                 if (complexVariant && 
#                     !variants.has(complexVariant) && 
#                     complexVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                     variants.add(complexVariant);
#                 }
                
#                 // Also create variant with base text + last group content
#                 if (!baseTextAdded) {
#                     const baseVariantParts = [
#                         baseText, 
#                         bracketGroups[bracketGroups.length-1].matches[0].content
#                     ];
                    
#                     const baseVariant = removeArticles(normalizeWhitespace(baseVariantParts.join(' ')));
                    
#                     if (baseVariant && 
#                         !variants.has(baseVariant) && 
#                         baseVariant.toUpperCase() !== removeArticles(originalTitle.toUpperCase())) {
#                         variants.add(baseVariant);
#                     }
#                 }
#             }
#         }
        
#         return Array.from(variants).sort();
#     $$;           
#     """).collect()
    
#     return


# def process_batches_sql(session, table_name):
#     """Process title variants using SQL and the registered JavaScript UDFs"""
#     print("Processing title variants using SQL...")
    
#     # Create a temporary table to store results
#     session.sql(f"""
#     CREATE OR REPLACE TEMPORARY TABLE EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS AS
#     SELECT
#         t.*,
#         t.TITLE as ORIGINAL_TITLE,
#         NULL as TITLE_VARIANT,
#         FALSE as IS_VARIANT
#     FROM {table_name} t
#     """).collect()
    
#     # Process variants using the JavaScript UDF and insert into the temporary table
#     session.sql(f"""
#     INSERT INTO EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS
#     SELECT 
#         t.*,
#         t.TITLE as ORIGINAL_TITLE,
#         v.value::STRING as TITLE_VARIANT,
#         TRUE as IS_VARIANT
#     FROM {table_name} t,
#     TABLE(FLATTEN(EDW_APPS.MATCHING.GENERATE_TITLE_VARIANTS_JS(t.TITLE))) v
#     WHERE v.value IS NOT NULL
#     """).collect()
    
#     # Update original rows to include their TITLE as TITLE_VARIANT if they have no variants
#     session.sql("""
#     UPDATE EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS orig
#     SET TITLE_VARIANT = TITLE
#     WHERE IS_VARIANT = FALSE
#     AND NOT EXISTS (
#         SELECT 1 FROM EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS var
#         WHERE var.ORIGINAL_TITLE = orig.TITLE
#         AND var.IS_VARIANT = TRUE
#     )
#     """).collect()
    
#     # Create the final table with ROW_NUMBER column
#     session.sql("""
#     CREATE OR REPLACE TABLE EDW_APPS.MATCHING.ADC_WORKS_TITLE_VARIANTS AS
#     SELECT 
#         ROW_NUMBER() OVER (ORDER BY ORIGINAL_TITLE, IS_VARIANT) as ROW_NUMBER,
#         *
#     FROM EDW_APPS.MATCHING.TEMP_TITLE_VARIANTS
#     //WHERE TITLE_VARIANT IS NOT NULL
#     """).collect()
    
#     stats = session.sql("""
#     SELECT
#         SUM(CASE WHEN IS_VARIANT = FALSE THEN 1 ELSE 0 END) as ORIGINAL_COUNT,
#         SUM(CASE WHEN IS_VARIANT = TRUE THEN 1 ELSE 0 END) as VARIANT_COUNT,
#         COUNT(*) as TOTAL_COUNT
#     FROM EDW_APPS.MATCHING.ADC_WORKS_TITLE_VARIANTS
#     """).collect()
    
#     print(f"OAriginal tracks: {stats[0]['ORIGINAL_COUNT']}")
#     print(f"Generated variants: {stats[0]['VARIANT_COUNT']}")
#     print(f"Total rows in output: {stats[0]['TOTAL_COUNT']}")
    
#     return

# def main(session):
#     # Get the input data from ADCWORKS
#     print("Reading data from ADCWORKS...")
#     table_name = "EDW_APPS.MATCHING.ADCWORKS"
    
#     # Register JavaScript UDFs for title variant generation
#     register_udfs(session)
    
#     # Process data in SQL batches
#     process_batches_sql(session, table_name)

    
#     print("\nTitle variant processing complete!")
#     return session.table("EDW_APPS.MATCHING.ADC_WORKS_TITLE_VARIANTS")

# # Run the main function
# result_df = main(session)

In [ ]:
--PRE_PARTITIONING

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION_CJ AS
-- SELECT 
--     *,
--     ROW_NUMBER() OVER (ORDER BY ROW_NUMBER) AS row_id,
--     MOD(ROW_NUMBER() OVER (ORDER BY ROW_NUMBER), 483000) AS partition_id
-- FROM ADC_WORKS_TITLE_VARIANTS
-- WHERE TITLE_VARIANT IS NOT NULL;

In [ ]:



-- -- Create a stored procedure to generate 100 partition tables
-- CREATE OR REPLACE PROCEDURE CREATE_PARTITION_TABLES()
-- RETURNS STRING
-- LANGUAGE JAVASCRIPT
-- AS
-- $$
--     var totalPartitions = 100;
--     var baseTableName = "ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION";
--     var partitionColumnName = "partition_id";
--     var targetSchemaTable = "ADC_WORKS_TITLE_VARIANTS_PART";
--     var successCount = 0;
--     var errors = [];
    
--     // Loop through all partition IDs from 0 to 99
--     for (var i = 0; i < totalPartitions; i++) {
--         try {
--             // Create the partition table
--             var createTableSQL = `
--                 CREATE OR REPLACE TABLE ${targetSchemaTable}_${i} AS
--                 SELECT * 
--                 FROM ${baseTableName}
--                 WHERE ${partitionColumnName} = ${i}
--             `;
            
--             // Execute the SQL statement
--             snowflake.execute({sqlText: createTableSQL});
            
--             // Verify table was created and has data
--             var verifySQL = `
--                 SELECT COUNT(*) AS record_count
--                 FROM ${targetSchemaTable}_${i}
--             `;
--             var result = snowflake.createStatement({sqlText: verifySQL}).execute();
--             result.next();
--             var recordCount = result.getColumnValue(1);
            
--             // Log progress
--             snowflake.execute({
--                 sqlText: `
--                     INSERT INTO partition_creation_log (
--                         partition_id, status, record_count, creation_time
--                     ) VALUES (?, 'SUCCESS', ?, CURRENT_TIMESTAMP())
--                 `,
--                 binds: [i, recordCount]
--             });
            
--             successCount++;
--         } catch (err) {
--             // Log any errors
--             errors.push(`Error creating partition ${i}: ${err}`);
            
--             snowflake.execute({
--                 sqlText: `
--                     INSERT INTO partition_creation_log (
--                         partition_id, status, error_message, creation_time
--                     ) VALUES (?, 'ERROR', ?, CURRENT_TIMESTAMP())
--                 `,
--                 binds: [i, err.toString()]
--             });
--         }
--     }
    
--     return `Created ${successCount} partition tables successfully. Errors: ${errors.length}`;
-- $$;

-- -- Create log table for tracking creation status
-- CREATE OR REPLACE TABLE partition_creation_log (
--     partition_id INTEGER,
--     status VARCHAR,
--     record_count INTEGER,
--     error_message VARCHAR,
--     creation_time TIMESTAMP_LTZ
-- );

-- -- Execute the procedure
-- CALL CREATE_PARTITION_TABLES();

-- -- Check results
-- SELECT * FROM partition_creation_log ORDER BY partition_id;

In [ ]:
-- -- Create a stored procedure to generate ONLY the first 5 partition tables


-- CREATE OR REPLACE PROCEDURE CREATE_PARTITION_TABLES()
-- RETURNS STRING
-- LANGUAGE JAVASCRIPT
-- AS
-- $$
--     // Change from 100 to 5 here
--     var totalPartitions = 5;
--     var baseTableName = "ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION_CJ";
--     var partitionColumnName = "partition_id";
--     var targetSchemaTable = "ADC_WORKS_TITLE_VARIANTS_PART";
--     var successCount = 0;
--     var errors = [];
    
--     // Loop through only partition IDs from 0 to 4
--     for (var i = 0; i < totalPartitions; i++) {
--         try {
--             // Create the partition table
--             var createTableSQL = `
--                 CREATE OR REPLACE TABLE ${targetSchemaTable}_${i} AS
--                 SELECT * 
--                 FROM ${baseTableName}
--                 WHERE ${partitionColumnName} = ${i}
--             `;
            
--             // Execute the SQL statement
--             snowflake.execute({sqlText: createTableSQL});
            
--             // Verify table was created and has data
--             var verifySQL = `
--                 SELECT COUNT(*) AS record_count
--                 FROM ${targetSchemaTable}_${i}
--             `;
--             var result = snowflake.createStatement({sqlText: verifySQL}).execute();
--             result.next();
--             var recordCount = result.getColumnValue(1);
            
--             // Log progress
--             snowflake.execute({
--                 sqlText: `
--                     INSERT INTO partition_creation_log (
--                         partition_id, status, record_count, creation_time
--                     ) VALUES (?, 'SUCCESS', ?, CURRENT_TIMESTAMP())
--                 `,
--                 binds: [i, recordCount]
--             });
            
--             successCount++;
--         } catch (err) {
--             // Log any errors
--             errors.push(`Error creating partition ${i}: ${err}`);
            
--             snowflake.execute({
--                 sqlText: `
--                     INSERT INTO partition_creation_log (
--                         partition_id, status, error_message, creation_time
--                     ) VALUES (?, 'ERROR', ?, CURRENT_TIMESTAMP())
--                 `,
--                 binds: [i, err.toString()]
--             });
--         }
--     }
    
--     return `Created ${successCount} partition tables successfully. Errors: ${errors.length}`;
-- $$;

-- -- Create log table for tracking creation status
-- CREATE OR REPLACE TABLE partition_creation_log (
--     partition_id INTEGER,
--     status VARCHAR,
--     record_count INTEGER,
--     error_message VARCHAR,
--     creation_time TIMESTAMP_LTZ
-- );

-- -- Execute the procedure
-- CALL CREATE_PARTITION_TABLES();

-- -- Check results
-- SELECT * FROM partition_creation_log ORDER BY partition_id;

In [ ]:
-- -- Create 15 partition tables


-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_0 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 0;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_1 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 1;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_2 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 2;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_3 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 3;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_4 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 4;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_5 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 5;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_6 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 6;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_7 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 7;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_8 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 8;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_9 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 9;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_10 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 10;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_11 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 11;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_12 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 12;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_13 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 13;

-- CREATE OR REPLACE TABLE ADC_WORKS_TITLE_VARIANTS_PART_14 AS
-- SELECT * FROM ADC_WORKS_TITLE_VARIANTS_WITH_PARTITION WHERE partition_id = 14;




In [ ]:

#setup_env


# import snowflake.snowpark as snowpark
# from snowflake.snowpark.functions import col, lit, udf, call_udf, expr
# import pandas as pd
# import time

# # Define view names and result table
# MAZOOKA_TRACKS_VIEW = "MZK_TRACKS_TITLE_VARIANTS" 
# RESULTS_TABLE = "EDW_APPS.MATCHING.TITLE_VARIANT_MATCHING_RESULTS"

# def create_jaro_winkler_similarity_udf(session):
#     """Create JavaScript UDF for title similarity calculation using Jaro-Winkler distance"""
#     try:
#         print("Creating title similarity function with Jaro-Winkler algorithm...")
        
#         # Create the JavaScript UDF using dollar-quoting to avoid escape issues
#         title_similarity_js = """
#         CREATE OR REPLACE FUNCTION TITLE_SIMILARITY(str1 STRING, str2 STRING)
#         RETURNS FLOAT
#         LANGUAGE JAVASCRIPT
#         AS 
#         $$
#           // Normalize inputs - uppercase, remove extra spaces, special chars, etc.
#           function normalizeString(str) {
#             if (!str) return '';
#             return str.toUpperCase()
#                       .replace(/[^A-Z0-9\\s]/g, ' ')  // Replace special chars with space
#                       .replace(/\\s+/g, ' ')          // Replace multiple spaces with single space
#                       .trim();
#           }
          
#           // Jaro similarity implementation
#           function jaroSimilarity(s1, s2) {
#             // If the strings are equal
#             if (s1 === s2) return 1.0;
            
#             // If either string is empty
#             if (s1.length === 0 || s2.length === 0) return 0.0;
            
#             // Maximum distance allowed for matching
#             const matchDistance = Math.floor(Math.max(s1.length, s2.length) / 2) - 1;
            
#             // Arrays to track matches
#             const s1Matches = Array(s1.length).fill(false);
#             const s2Matches = Array(s2.length).fill(false);
            
#             // Count of matches
#             let matches = 0;
            
#             // Look for matches
#             for (let i = 0; i < s1.length; i++) {
#               // Lower and upper bound for matching
#               const start = Math.max(0, i - matchDistance);
#               const end = Math.min(i + matchDistance + 1, s2.length);
              
#               for (let j = start; j < end; j++) {
#                 // Skip if already matched or not matching
#                 if (s2Matches[j] || s1[i] !== s2[j]) continue;
                
#                 // Found a match
#                 s1Matches[i] = true;
#                 s2Matches[j] = true;
#                 matches++;
#                 break;
#               }
#             }
            
#             // If no matches, return 0
#             if (matches === 0) return 0.0;
            
#             // Count transpositions
#             let transpositions = 0;
#             let k = 0;
            
#             for (let i = 0; i < s1.length; i++) {
#               if (!s1Matches[i]) continue;
              
#               while (!s2Matches[k]) k++;
              
#               if (s1[i] !== s2[k]) transpositions++;
              
#               k++;
#             }
            
#             // Calculate Jaro similarity
#             const jaroSim = (
#               (matches / s1.length) +
#               (matches / s2.length) +
#               ((matches - transpositions / 2) / matches)
#             ) / 3;
            
#             return jaroSim;
#           }
          
#           // Jaro-Winkler similarity
#           function jaroWinklerSimilarity(s1, s2) {
#             const normalized1 = normalizeString(s1);
#             const normalized2 = normalizeString(s2);
            
#             // Calculate Jaro similarity
#             const jaroSim = jaroSimilarity(normalized1, normalized2);
            
#             // Calculate prefix length (max 4)
#             let prefixLength = 0;
#             const maxPrefixLength = Math.min(4, Math.min(normalized1.length, normalized2.length));
            
#             for (let i = 0; i < maxPrefixLength; i++) {
#               if (normalized1[i] === normalized2[i]) {
#                 prefixLength++;
#               } else {
#                 break;
#               }
#             }
            
#             // Scaling factor for how much the score is adjusted by prefix length
#             const scalingFactor = 0.1;
            
#             // Calculate Jaro-Winkler similarity
#             return jaroSim + (prefixLength * scalingFactor * (1 - jaroSim));
#           }
          
#           // Main entry point for the UDF
#           return jaroWinklerSimilarity(STR1, STR2);
#         $$
#         """
        
#         # Execute the SQL statement
#         session.sql(title_similarity_js).collect()
#         print("Title similarity UDF with Jaro-Winkler distance created successfully.")
#         return True
#     except Exception as e:
#         print(f"Error creating title similarity UDF: {e}")
#         return False

# def prepare_tracks_variant_view(session):
#     """Create optimized enhanced view of title variants from Mazooka tracks table"""
#     try:
#         # Create enhanced Mazooka Tracks view with normalized titles - all uppercase
#         tracks_view_sql = f"""
#         CREATE OR REPLACE TEMPORARY VIEW TRACKS_VARIANTS AS
#         SELECT
#             TRACK_ID,
#             UPPER(TITLE) AS TITLE,
#             UPPER(TITLE_VARIANT) AS TITLE_VARIANT,
#             UPPER(ORIGINAL_TITLE) AS ORIGINAL_TITLE,
#             UPPER(ISWC) AS ISWC,
#             IS_VARIANT,
#             UPPER(TITLE) AS UPPER_TITLE,
#             UPPER(TITLE_VARIANT) AS UPPER_TITLE_VARIANT,
#             UPPER(ORIGINAL_TITLE) AS UPPER_ORIGINAL_TITLE,
#             UPPER(REGEXP_REPLACE(TITLE, '[^A-Z0-9\\\\s]', ' ')) AS CLEAN_TITLE,
#             UPPER(REGEXP_REPLACE(TITLE, '[^A-Z0-9]', '')) AS COMPACT_TITLE,
#             UPPER(REGEXP_REPLACE(REGEXP_REPLACE(TITLE, '[^A-Z0-9\\\\s]', ' '), '^(THE|A|AN)\\\\s+', '')) AS NO_ARTICLE_TITLE
#         FROM {MAZOOKA_TRACKS_VIEW}
#         """
        
#         # Execute the SQL statements
#         session.sql(tracks_view_sql).collect()
        
#         print("Created enhanced view for Mazooka tracks with normalized uppercase titles")
#         return True
#     except Exception as e:
#         print(f"Error creating enhanced tracks view: {e}")
#         return False

# def create_results_table(session):
#     """Create the results table structure to store matches"""
#     try:
#         create_table_sql = f"""
#         CREATE TABLE IF NOT EXISTS {RESULTS_TABLE} (
#             APRA_WORK_ID VARCHAR,
#             APRA_TITLE VARCHAR,
#             MUZOOKA_TRACK_ID VARCHAR,
#             MUZOOKA_TITLE VARCHAR,
#             TITLE_MATCH_SCORE FLOAT,
#             APRA_ISWC VARCHAR,
#             MUZOOKA_ISWC VARCHAR,
#             PARTITION_ID INTEGER
#         )
#         """
#         session.sql(create_table_sql).collect()
#         print(f"Created or verified results table: {RESULTS_TABLE}")
#         return True
#     except Exception as e:
#         print(f"Error creating results table: {e}")
#         return False

# def setup_environment():
#     """Setup the environment for all partitions"""
#     # Create Snowflake session
#     session = snowpark.Session.builder.getOrCreate()
    
#     # Create the similarity UDF
#     create_jaro_winkler_similarity_udf(session)
    
#     # Create the tracks variants view
#     prepare_tracks_variant_view(session)
    
#     # Create the results table
#     create_results_table(session)
    
#     print("Environment setup complete!")
    
#     return session

# if __name__ == "__main__":
#     # Setup environment
#     session = setup_environment()
    
#     print("Environment setup complete. Run individual partition matching scripts next.")

In [ ]:
# # partition_n_title_variant_matcher.py

# import snowflake.snowpark as snowpark
# import time

# PARTITION_ID = 0  # Change for each partition (0-99)
# ADC_PARTITION_TABLE = f"ADC_WORKS_TITLE_VARIANTS_PART_{PARTITION_ID}"
# RESULTS_TABLE = f"EDW_APPS.MATCHING.TITLE_VARIANT_PARTITION_{PARTITION_ID}_MATCHING_RESULTS"
# BATCH_SIZE = 10000

# def process_partition():
#     print(f"Starting processing for Partition {PARTITION_ID} - TITLE VARIANTS ONLY (NO ISWC MATCHING)...")
    
#     # Create Snowflake session
#     session = snowpark.Session.builder.getOrCreate()
    
#     # Create a temporary results table for this partition
#     partition_results_table = f"TEMP_VARIANT_MATCHES_PARTITION_{PARTITION_ID}"
#     session.sql(f"""
#     CREATE OR REPLACE TEMPORARY TABLE {partition_results_table} (
#         APRA_WORK_ID VARCHAR,
#         APRA_TITLE VARCHAR,
#         APRA_TITLE_VARIANT VARCHAR,
#         MUZOOKA_TRACK_ID VARCHAR,
#         MUZOOKA_TITLE VARCHAR,
#         MUZOOKA_TITLE_VARIANT VARCHAR,
#         TITLE_VARIANT_MATCH_SCORE FLOAT,
#         APRA_ISWC VARCHAR,
#         MUZOOKA_ISWC VARCHAR,
#         MATCH_TYPE VARCHAR
#     )
#     """).collect()
    
#     # Create enhanced ADC view for this partition - focusing on title variants
#     adc_view_sql = f"""
#     CREATE OR REPLACE TEMPORARY VIEW ADC_VARIANTS_{PARTITION_ID} AS
#     SELECT
#         APRA_WORK_ID,
#         UPPER(TITLE) AS TITLE,
#         UPPER(TITLE_VARIANT) AS TITLE_VARIANT,
#         UPPER(ORIGINAL_TITLE) AS ORIGINAL_TITLE,
#         UPPER(ISWC) AS ISWC,
#         IS_VARIANT,
#         -- Normalize title variants for matching
#         UPPER(TITLE_VARIANT) AS UPPER_TITLE_VARIANT,
#         UPPER(REGEXP_REPLACE(TITLE_VARIANT, '[^A-Z0-9\\\\s]', ' ')) AS CLEAN_TITLE_VARIANT,
#         UPPER(REGEXP_REPLACE(TITLE_VARIANT, '[^A-Z0-9]', '')) AS COMPACT_TITLE_VARIANT,
#         UPPER(REGEXP_REPLACE(REGEXP_REPLACE(TITLE_VARIANT, '[^A-Z0-9\\\\s]', ' '), '^(THE|A|AN)\\\\s+', '')) AS NO_ARTICLE_TITLE_VARIANT,
#         -- Add first character and length for pre-filtering
#         LEFT(UPPER(TITLE_VARIANT), 1) AS FIRST_CHAR,
#         LENGTH(TITLE_VARIANT) AS VARIANT_LENGTH
#     FROM {ADC_PARTITION_TABLE}
#     -- Only include records that have a title variant
#     WHERE TITLE_VARIANT IS NOT NULL AND TRIM(TITLE_VARIANT) != ''
#     """
#     session.sql(adc_view_sql).collect()
    
#     # Create temporary view for Muzooka tracks with title variants
#     tracks_view_sql = f"""
#     CREATE OR REPLACE TEMPORARY VIEW TRACKS_VARIANTS_{PARTITION_ID} AS
#     SELECT
#         TRACK_ID,
#         UPPER(TITLE) AS TITLE,
#         UPPER(TITLE_VARIANT) AS TITLE_VARIANT,
#         UPPER(ORIGINAL_TITLE) AS ORIGINAL_TITLE,
#         UPPER(ISWC) AS ISWC,
#         IS_VARIANT,
#         -- Normalize title variants for matching
#         UPPER(TITLE_VARIANT) AS UPPER_TITLE_VARIANT,
#         UPPER(REGEXP_REPLACE(TITLE_VARIANT, '[^A-Z0-9\\\\s]', ' ')) AS CLEAN_TITLE_VARIANT,
#         UPPER(REGEXP_REPLACE(TITLE_VARIANT, '[^A-Z0-9]', '')) AS COMPACT_TITLE_VARIANT,
#         UPPER(REGEXP_REPLACE(REGEXP_REPLACE(TITLE_VARIANT, '[^A-Z0-9\\\\s]', ' '), '^(THE|A|AN)\\\\s+', '')) AS NO_ARTICLE_TITLE_VARIANT,
#         -- Add first character and length for pre-filtering
#         LEFT(UPPER(TITLE_VARIANT), 1) AS FIRST_CHAR,
#         LENGTH(TITLE_VARIANT) AS VARIANT_LENGTH
#     FROM MZK_TRACKS_TITLE_VARIANTS
#     -- Only include records that have a title variant
#     WHERE TITLE_VARIANT IS NOT NULL AND TRIM(TITLE_VARIANT) != ''
#     """
#     session.sql(tracks_view_sql).collect()
    
#     # Get record count
#     adc_count = session.sql(f"SELECT COUNT(*) as count FROM ADC_VARIANTS_{PARTITION_ID}").collect()[0]['COUNT']
#     print(f"Partition {PARTITION_ID} has {adc_count} records with title variants to process")
    
#     # Calculate number of batches
#     total_batches = -(-adc_count // BATCH_SIZE)  # Ceiling division
    
#     # Process in batches
#     total_processed = 0
#     start_time = time.time()
    
#     for batch_num in range(total_batches):
#         batch_start_time = time.time()
#         batch_start = batch_num * BATCH_SIZE
        
#         # Create and populate batch table
#         session.sql(f"""
#         CREATE OR REPLACE TEMPORARY TABLE BATCH_{PARTITION_ID} AS
#         SELECT *
#         FROM ADC_VARIANTS_{PARTITION_ID}
#         ORDER BY APRA_WORK_ID
#         LIMIT {BATCH_SIZE} OFFSET {batch_start}
#         """).collect()
        
#         # Count records in this batch
#         records_in_batch = session.sql(f"SELECT COUNT(*) as count FROM BATCH_{PARTITION_ID}").collect()[0]['COUNT']
        
#         # Skip if empty batch
#         if records_in_batch == 0:
#             break
        
#         # Process matching for this batch - ONLY MATCHING TITLE VARIANTS WITH PRE-FILTERING
#         batch_match_sql = f"""
#         INSERT INTO {partition_results_table}
#         SELECT 
#             a.APRA_WORK_ID,
#             a.TITLE AS APRA_TITLE,
#             a.TITLE_VARIANT AS APRA_TITLE_VARIANT,
#             t.TRACK_ID AS MUZOOKA_TRACK_ID,
#             t.TITLE AS MUZOOKA_TITLE,
#             t.TITLE_VARIANT AS MUZOOKA_TITLE_VARIANT,
#             GREATEST(
#                 -- Calculate exact match scores between title variants
#                 CASE 
#                     WHEN a.UPPER_TITLE_VARIANT = t.UPPER_TITLE_VARIANT THEN 1.0
#                     WHEN a.CLEAN_TITLE_VARIANT = t.CLEAN_TITLE_VARIANT THEN 0.95
#                     WHEN a.COMPACT_TITLE_VARIANT = t.COMPACT_TITLE_VARIANT THEN 0.90
#                     WHEN a.NO_ARTICLE_TITLE_VARIANT = t.NO_ARTICLE_TITLE_VARIANT THEN 0.85
#                     ELSE TITLE_SIMILARITY(a.TITLE_VARIANT, t.TITLE_VARIANT)
#                 END
#             ) AS TITLE_VARIANT_MATCH_SCORE,
#             a.ISWC AS APRA_ISWC,
#             t.ISWC AS MUZOOKA_ISWC,
#             'VARIANT_MATCH' AS MATCH_TYPE
#         FROM 
#             BATCH_{PARTITION_ID} a
#         JOIN 
#             TRACKS_VARIANTS_{PARTITION_ID} t
#         ON
#             -- Pre-filtering to reduce cross join size
#             a.FIRST_CHAR = t.FIRST_CHAR AND
#             ABS(a.VARIANT_LENGTH - t.VARIANT_LENGTH) <= 5
#         WHERE
#             -- Filter combinations for better performance - ONLY COMPARE VARIANTS
#             a.UPPER_TITLE_VARIANT = t.UPPER_TITLE_VARIANT OR
#             a.CLEAN_TITLE_VARIANT = t.CLEAN_TITLE_VARIANT OR
#             a.COMPACT_TITLE_VARIANT = t.COMPACT_TITLE_VARIANT OR
#             a.NO_ARTICLE_TITLE_VARIANT = t.NO_ARTICLE_TITLE_VARIANT OR
#             TITLE_SIMILARITY(a.TITLE_VARIANT, t.TITLE_VARIANT) >= 0.8
#         """
#         session.sql(batch_match_sql).collect()
        
#         # Update processed count
#         total_processed += records_in_batch
        
#         # Report progress
#         batch_matches = session.sql(f"""
#             SELECT COUNT(*) as count FROM {partition_results_table}
#             WHERE APRA_WORK_ID IN (SELECT APRA_WORK_ID FROM BATCH_{PARTITION_ID})
#         """).collect()[0]['COUNT']
        
#         batch_time = time.time() - batch_start_time
#         percent_complete = (total_processed / adc_count) * 100 if adc_count > 0 else 100
        
#         print(f"Partition {PARTITION_ID} - Batch {batch_num + 1}/{total_batches}: {records_in_batch} records processed")
#         print(f"Found {batch_matches} title variant matches in {batch_time:.2f} seconds")
#         print(f"Progress: {total_processed}/{adc_count} records ({percent_complete:.2f}%)")
    
#     # Insert final results
#     print(f"Partition {PARTITION_ID} - All batches processed. Inserting into final results table...")
    
#     # Create the results table if it doesn't exist
#     session.sql(f"""
#     CREATE TABLE IF NOT EXISTS {RESULTS_TABLE} (
#         APRA_WORK_ID VARCHAR,
#         APRA_TITLE VARCHAR,
#         APRA_TITLE_VARIANT VARCHAR,
#         MUZOOKA_TRACK_ID VARCHAR,
#         MUZOOKA_TITLE VARCHAR,
#         MUZOOKA_TITLE_VARIANT VARCHAR,
#         TITLE_VARIANT_MATCH_SCORE FLOAT,
#         APRA_ISWC VARCHAR,
#         MUZOOKA_ISWC VARCHAR,
#         PARTITION_ID NUMBER,
#         MATCH_TYPE VARCHAR
#     )
#     """).collect()
    
#     # Clean existing data for this partition
#     session.sql(f"""
#     DELETE FROM {RESULTS_TABLE}
#     WHERE PARTITION_ID = {PARTITION_ID}
#     """).collect()
    
#     # Insert results
#     session.sql(f"""
#     INSERT INTO {RESULTS_TABLE}
#     SELECT 
#         APRA_WORK_ID,
#         APRA_TITLE,
#         APRA_TITLE_VARIANT,
#         MUZOOKA_TRACK_ID,
#         MUZOOKA_TITLE,
#         MUZOOKA_TITLE_VARIANT,
#         TITLE_VARIANT_MATCH_SCORE,
#         APRA_ISWC,
#         MUZOOKA_ISWC,
#         {PARTITION_ID} AS PARTITION_ID,
#         MATCH_TYPE
#     FROM {partition_results_table}
#     WHERE TITLE_VARIANT_MATCH_SCORE >= 0.8
#     """).collect()
    
#     # Get final counts
#     final_matches = session.sql(f"""
#     SELECT COUNT(*) as count FROM {RESULTS_TABLE}
#     WHERE PARTITION_ID = {PARTITION_ID}
#     """).collect()[0]['COUNT']
    
#     # Get stats on match scores
#     match_stats = session.sql(f"""
#     SELECT 
#         COUNT(*) AS total_matches,
#         SUM(CASE WHEN TITLE_VARIANT_MATCH_SCORE = 1.0 THEN 1 ELSE 0 END) AS exact_matches,
#         SUM(CASE WHEN TITLE_VARIANT_MATCH_SCORE >= 0.95 AND TITLE_VARIANT_MATCH_SCORE < 1.0 THEN 1 ELSE 0 END) AS clean_matches,
#         SUM(CASE WHEN TITLE_VARIANT_MATCH_SCORE >= 0.9 AND TITLE_VARIANT_MATCH_SCORE < 0.95 THEN 1 ELSE 0 END) AS compact_matches,
#         SUM(CASE WHEN TITLE_VARIANT_MATCH_SCORE >= 0.85 AND TITLE_VARIANT_MATCH_SCORE < 0.9 THEN 1 ELSE 0 END) AS no_article_matches,
#         SUM(CASE WHEN TITLE_VARIANT_MATCH_SCORE >= 0.8 AND TITLE_VARIANT_MATCH_SCORE < 0.85 THEN 1 ELSE 0 END) AS fuzzy_matches,
#         MIN(TITLE_VARIANT_MATCH_SCORE) AS min_score,
#         MAX(TITLE_VARIANT_MATCH_SCORE) AS max_score,
#         AVG(TITLE_VARIANT_MATCH_SCORE) AS avg_score
#     FROM {RESULTS_TABLE}
#     WHERE PARTITION_ID = {PARTITION_ID}
#     """).collect()[0]
    
#     total_time = time.time() - start_time
#     print(f"Partition {PARTITION_ID} complete!")
#     print(f"Processed {adc_count} records in {total_time:.2f} seconds")
#     print(f"Found {final_matches} total title variant matches with score >= 0.8")
#     print(f"Match types breakdown:")
#     print(f"  - Exact matches (1.0): {match_stats['EXACT_MATCHES']}")
#     print(f"  - Clean matches (0.95-0.99): {match_stats['CLEAN_MATCHES']}")
#     print(f"  - Compact matches (0.90-0.94): {match_stats['COMPACT_MATCHES']}")
#     print(f"  - No-article matches (0.85-0.89): {match_stats['NO_ARTICLE_MATCHES']}")
#     print(f"  - Fuzzy matches (0.80-0.84): {match_stats['FUZZY_MATCHES']}")
#     print(f"Score range: {match_stats['MIN_SCORE']:.2f} - {match_stats['MAX_SCORE']:.2f}, Average: {match_stats['AVG_SCORE']:.2f}")
    
#     # Clean up temporary objects
#     session.sql(f"DROP TABLE IF EXISTS BATCH_{PARTITION_ID}").collect()
#     session.sql(f"DROP VIEW IF EXISTS ADC_VARIANTS_{PARTITION_ID}").collect()
#     session.sql(f"DROP VIEW IF EXISTS TRACKS_VARIANTS_{PARTITION_ID}").collect()
#     session.sql(f"DROP TABLE IF EXISTS {partition_results_table}").collect()
    
#     return adc_count, final_matches, total_time

# if __name__ == "__main__":
#     # Create Snowflake session
#     session = snowpark.Session.builder.getOrCreate()
    
#     # Create the similarity UDF if it doesn't exist yet
#     try:
#         session.sql("SELECT TITLE_SIMILARITY('test', 'test')").collect()
#         print("Title similarity UDF already exists.")
#     except Exception as e:
#         print("Creating title similarity UDF...")
#         title_similarity_js = """
#         CREATE OR REPLACE FUNCTION TITLE_SIMILARITY(str1 STRING, str2 STRING)
#         RETURNS FLOAT
#         LANGUAGE JAVASCRIPT
#         AS 
#         $$
#           // Normalize inputs - uppercase, remove extra spaces, special chars, etc.
#           function normalizeString(str) {
#             if (!str) return '';
#             return str.toUpperCase()
#                       .replace(/[^A-Z0-9\\s]/g, ' ')  // Replace special chars with space
#                       .replace(/\\s+/g, ' ')          // Replace multiple spaces with single space
#                       .trim();
#           }
          
#           // Jaro-Winkler similarity implementation
#           function jaroWinklerSimilarity(s1, s2) {
#             const normalized1 = normalizeString(s1);
#             const normalized2 = normalizeString(s2);
            
#             // If the strings are equal
#             if (normalized1 === normalized2) return 1.0;
            
#             // If either string is empty
#             if (normalized1.length === 0 || normalized2.length === 0) return 0.0;
            
#             // Maximum distance allowed for matching
#             const matchDistance = Math.floor(Math.max(normalized1.length, normalized2.length) / 2) - 1;
            
#             // Arrays to track matches
#             const s1Matches = Array(normalized1.length).fill(false);
#             const s2Matches = Array(normalized2.length).fill(false);
            
#             // Count of matches
#             let matches = 0;
            
#             // Look for matches
#             for (let i = 0; i < normalized1.length; i++) {
#               // Lower and upper bound for matching
#               const start = Math.max(0, i - matchDistance);
#               const end = Math.min(i + matchDistance + 1, normalized2.length);
              
#               for (let j = start; j < end; j++) {
#                 // Skip if already matched or not matching
#                 if (s2Matches[j] || normalized1[i] !== normalized2[j]) continue;
                
#                 // Found a match
#                 s1Matches[i] = true;
#                 s2Matches[j] = true;
#                 matches++;
#                 break;
#               }
#             }
            
#             // If no matches, return 0
#             if (matches === 0) return 0.0;
            
#             // Count transpositions
#             let transpositions = 0;
#             let k = 0;
            
#             for (let i = 0; i < normalized1.length; i++) {
#               if (!s1Matches[i]) continue;
              
#               while (!s2Matches[k]) k++;
              
#               if (normalized1[i] !== normalized2[k]) transpositions++;
              
#               k++;
#             }
            
#             // Calculate Jaro similarity
#             const jaroSim = (
#               (matches / normalized1.length) +
#               (matches / normalized2.length) +
#               ((matches - transpositions / 2) / matches)
#             ) / 3;
            
#             // Calculate prefix length (max 4)
#             let prefixLength = 0;
#             const maxPrefixLength = Math.min(4, Math.min(normalized1.length, normalized2.length));
            
#             for (let i = 0; i < maxPrefixLength; i++) {
#               if (normalized1[i] === normalized2[i]) {
#                 prefixLength++;
#               } else {
#                 break;
#               }
#             }
            
#             // Scaling factor for how much the score is adjusted by prefix length
#             const scalingFactor = 0.1;
            
#             // Calculate Jaro-Winkler similarity
#             return jaroSim + (prefixLength * scalingFactor * (1 - jaroSim));
#           }
          
#           // Main entry point for the UDF
#           return jaroWinklerSimilarity(STR1, STR2);
#         $$
#         """
#         session.sql(title_similarity_js).collect()
#         print("Title similarity UDF created successfully.")
    
#     # Process the partition
#     process_partition()

In [ ]:
# import snowflake.snowpark as snowpark
# import time

# # Define constants
# PARTITION_ID = 0
# ADC_PARTITION_TABLE = f"ADC_WORKS_TITLE_VARIANTS_PART_{PARTITION_ID}"
# RESULTS_TABLE = f"EDW_APPS.MATCHING.TITLE_VARIANT_PARTITION_{PARTITION_ID}_MATCHING_RESULTS"
# BATCH_SIZE = 10000

# def process_partition():
#     print(f"Starting processing for Partition {PARTITION_ID} - VARIANT ONLY MATCHING WITH FORCED UPPERCASE...")
#     start_time = time.time()
    
#     # First, let's update any existing data to uppercase
#     try:
#         print(f"Attempting to update existing results to uppercase...")
#         session.sql(f"""
#         UPDATE {RESULTS_TABLE}
#         SET 
#             APRA_TITLE = UPPER(APRA_TITLE),
#             APRA_TITLE_VARIANT = UPPER(APRA_TITLE_VARIANT),
#             MUZOOKA_TITLE = UPPER(MUZOOKA_TITLE),
#             MUZOOKA_TITLE_VARIANT = UPPER(MUZOOKA_TITLE_VARIANT),
#             APRA_ISWC = UPPER(APRA_ISWC),
#             MUZOOKA_ISWC = UPPER(MUZOOKA_ISWC),
#             MATCH_TYPE = UPPER(MATCH_TYPE)
#         WHERE PARTITION_ID = {PARTITION_ID}
#         """).collect()
#         print(f"Updated existing results to uppercase.")
#     except Exception as e:
#         print(f"Could not update existing results: {e}")
    
#     # Drop all existing results for this partition to start fresh
#     try:
#         print(f"Deleting existing results for partition {PARTITION_ID}...")
#         session.sql(f"""
#         DELETE FROM {RESULTS_TABLE}
#         WHERE PARTITION_ID = {PARTITION_ID}
#         """).collect()
#         print(f"Deleted existing results.")
#     except Exception as e:
#         print(f"Error deleting results: {e}")
    
#     # First, let's check the structure of the existing results table
#     try:
#         desc_result = session.sql(f"DESC TABLE {RESULTS_TABLE}").collect()
#         columns = [row['name'] for row in desc_result]
#         print(f"Existing results table has columns: {columns}")
        
#         # Check if we need to recreate the table
#         if len(columns) != 11:  # If we don't have exactly the 11 columns we need
#             print(f"Recreating the results table to match our expected schema...")
#             session.sql(f"DROP TABLE IF EXISTS {RESULTS_TABLE}").collect()
#             recreate_table = True
#         else:
#             recreate_table = False
#     except Exception as e:
#         print(f"Table doesn't exist or can't be described: {e}")
#         recreate_table = True
    
#     # Create or recreate final results table if needed
#     if recreate_table:
#         print(f"Creating results table {RESULTS_TABLE}...")
#         session.sql(f"""
#         CREATE TABLE IF NOT EXISTS {RESULTS_TABLE} (
#             APRA_WORK_ID VARCHAR,
#             APRA_TITLE VARCHAR,
#             APRA_TITLE_VARIANT VARCHAR,  
#             MUZOOKA_TRACK_ID VARCHAR,
#             MUZOOKA_TITLE VARCHAR,
#             MUZOOKA_TITLE_VARIANT VARCHAR,  
#             TITLE_VARIANT_MATCH_SCORE FLOAT,
#             APRA_ISWC VARCHAR,
#             MUZOOKA_ISWC VARCHAR,
#             PARTITION_ID NUMBER,
#             MATCH_TYPE VARCHAR
#         )
#         """).collect()
    
#     # Create tables for storing results
#     iswc_matches_table = f"TEMP_ISWC_MATCHES_PARTITION_{PARTITION_ID}"
#     variant_matches_table = f"TEMP_VARIANT_MATCHES_PARTITION_{PARTITION_ID}"
    
#     # Create ISWC matches table - ensure schema matches results table
#     session.sql(f"""
#     CREATE OR REPLACE TEMPORARY TABLE {iswc_matches_table} (
#         APRA_WORK_ID VARCHAR,
#         APRA_TITLE VARCHAR,
#         APRA_TITLE_VARIANT VARCHAR,
#         MUZOOKA_TRACK_ID VARCHAR,
#         MUZOOKA_TITLE VARCHAR,
#         MUZOOKA_TITLE_VARIANT VARCHAR,
#         TITLE_VARIANT_MATCH_SCORE FLOAT,
#         APRA_ISWC VARCHAR,
#         MUZOOKA_ISWC VARCHAR,
#         PARTITION_ID NUMBER,
#         MATCH_TYPE VARCHAR
#     )
#     """).collect()
    
#     # Create variant matches table with the same structure
#     session.sql(f"""
#     CREATE OR REPLACE TEMPORARY TABLE {variant_matches_table} AS 
#     SELECT * FROM {iswc_matches_table} WHERE 1=0
#     """).collect()
    
#     # Create enhanced ADC view for this partition - focusing on title variants
#     adc_view_sql = f"""
#     CREATE OR REPLACE TEMPORARY VIEW ADC_VARIANTS_{PARTITION_ID} AS
#     SELECT
#         APRA_WORK_ID,
#         UPPER(TITLE) AS TITLE,
#         UPPER(TITLE_VARIANT) AS TITLE_VARIANT,
#         UPPER(ORIGINAL_TITLE) AS ORIGINAL_TITLE,
#         UPPER(REGEXP_REPLACE(ISWC, '[^A-Za-z0-9]', '')) AS ISWC,
#         IS_VARIANT,
#         -- Focus on processing the title variant
#         UPPER(TITLE_VARIANT) AS UPPER_TITLE_VARIANT,
#         UPPER(REGEXP_REPLACE(TITLE_VARIANT, '[^A-Z0-9\\\\s]', ' ')) AS CLEAN_TITLE_VARIANT,
#         UPPER(REGEXP_REPLACE(TITLE_VARIANT, '[^A-Z0-9]', '')) AS COMPACT_TITLE_VARIANT,
#         UPPER(REGEXP_REPLACE(REGEXP_REPLACE(TITLE_VARIANT, '[^A-Z0-9\\\\s]', ' '), '^(THE|A|AN)\\\\s+', '')) AS NO_ARTICLE_TITLE_VARIANT
#     FROM {ADC_PARTITION_TABLE}
#     -- Only include records that have a title variant
#     WHERE TITLE_VARIANT IS NOT NULL AND TRIM(TITLE_VARIANT) != ''
#     """
#     session.sql(adc_view_sql).collect()
    
#     # Get record count
#     adc_count = session.sql(f"SELECT COUNT(*) as count FROM ADC_VARIANTS_{PARTITION_ID}").collect()[0]['COUNT']
#     print(f"Partition {PARTITION_ID} has {adc_count} records with title variants to process")
    
#     # STEP 1: IDENTIFY RECORDS WITH MATCHING ISWCs
#     print("Step 1: Identifying records with matching ISWCs...")
    
#     # Create a table of records that have matching ISWCs in the Muzooka dataset
#     session.sql(f"""
#     CREATE OR REPLACE TEMPORARY TABLE ISWC_MATCHED_RECORDS_{PARTITION_ID} AS
#     SELECT 
#         a.APRA_WORK_ID,
#         a.TITLE AS APRA_TITLE,
#         a.TITLE_VARIANT AS APRA_TITLE_VARIANT,
#         a.UPPER_TITLE_VARIANT,
#         a.CLEAN_TITLE_VARIANT,
#         a.COMPACT_TITLE_VARIANT,
#         a.NO_ARTICLE_TITLE_VARIANT,
#         a.ISWC AS APRA_ISWC,
#         t.TRACK_ID AS MUZOOKA_TRACK_ID,
#         UPPER(t.TITLE) AS MUZOOKA_TITLE,  -- Force uppercase
#         UPPER(t.TITLE_VARIANT) AS MUZOOKA_TITLE_VARIANT,  -- Force uppercase
#         UPPER(REGEXP_REPLACE(t.ISWC, '[^A-Za-z0-9]', '')) AS MUZOOKA_ISWC
#     FROM 
#         ADC_VARIANTS_{PARTITION_ID} a
#     JOIN 
#         MZK_TRACKS_TITLE_VARIANTS t
#     ON
#         a.ISWC = UPPER(REGEXP_REPLACE(t.ISWC, '[^A-Za-z0-9]', ''))
#     -- Only include Muzooka records that have a title variant
#     WHERE t.TITLE_VARIANT IS NOT NULL AND TRIM(t.TITLE_VARIANT) != ''
#     """).collect()
    
#     # Count records with ISWC matches
#     iswc_match_count = session.sql(f"SELECT COUNT(DISTINCT APRA_WORK_ID) as count FROM ISWC_MATCHED_RECORDS_{PARTITION_ID}").collect()[0]['COUNT']
#     print(f"Found {iswc_match_count} records with matching ISWCs and valid title variants")
    
#     # STEP 2: CALCULATE TITLE VARIANT MATCH SCORES FOR ISWC-MATCHED RECORDS
#     print("Step 2: Calculating title variant match scores for ISWC-matched records...")
    
#     # Calculate match scores only for title variants and insert into ISWC matches table
#     session.sql(f"""
#     INSERT INTO {iswc_matches_table}
#     SELECT 
#         APRA_WORK_ID,
#         UPPER(APRA_TITLE) AS APRA_TITLE,  -- Extra UPPER to be absolutely sure
#         UPPER(APRA_TITLE_VARIANT) AS APRA_TITLE_VARIANT,  -- Extra UPPER to be absolutely sure
#         MUZOOKA_TRACK_ID,
#         UPPER(MUZOOKA_TITLE) AS MUZOOKA_TITLE,  -- Extra UPPER to be absolutely sure
#         UPPER(MUZOOKA_TITLE_VARIANT) AS MUZOOKA_TITLE_VARIANT,  -- Extra UPPER to be absolutely sure
#         GREATEST(
#             -- Calculate exact match scores between title variants
#             CASE 
#                 WHEN UPPER_TITLE_VARIANT = UPPER(MUZOOKA_TITLE_VARIANT) THEN 1.0
#                 WHEN CLEAN_TITLE_VARIANT = UPPER(REGEXP_REPLACE(MUZOOKA_TITLE_VARIANT, '[^A-Z0-9\\\\s]', ' ')) THEN 0.95
#                 WHEN COMPACT_TITLE_VARIANT = UPPER(REGEXP_REPLACE(MUZOOKA_TITLE_VARIANT, '[^A-Z0-9]', '')) THEN 0.90
#                 WHEN NO_ARTICLE_TITLE_VARIANT = UPPER(REGEXP_REPLACE(REGEXP_REPLACE(MUZOOKA_TITLE_VARIANT, '[^A-Z0-9\\\\s]', ' '), '^(THE|A|AN)\\\\s+', '')) THEN 0.85
#                 ELSE TITLE_SIMILARITY(APRA_TITLE_VARIANT, MUZOOKA_TITLE_VARIANT)
#             END
#         ) AS TITLE_VARIANT_MATCH_SCORE,
#         UPPER(APRA_ISWC) AS APRA_ISWC,  -- Extra UPPER to be absolutely sure
#         UPPER(MUZOOKA_ISWC) AS MUZOOKA_ISWC,  -- Extra UPPER to be absolutely sure
#         {PARTITION_ID} AS PARTITION_ID,
#         'ISWC_MATCH' AS MATCH_TYPE
#     FROM 
#         ISWC_MATCHED_RECORDS_{PARTITION_ID}
#     """).collect()
    
#     # Count records with good title variant matches among ISWC matches
#     good_iswc_variant_matches = session.sql(f"""
#     SELECT COUNT(*) as count FROM {iswc_matches_table}
#     WHERE TITLE_VARIANT_MATCH_SCORE >= 0.8
#     """).collect()[0]['COUNT']
    
#     print(f"Among ISWC matches, found {good_iswc_variant_matches} records with title variant match score >= 0.8")
    
#     # STEP 3: INSERT ISWC MATCHES INTO FINAL RESULTS TABLE
#     print("Step 3: Inserting ISWC matches into final results table...")
    
#     # Insert with an additional UPPER() for each string field to double-ensure uppercase in the results
#     session.sql(f"""
#     INSERT INTO {RESULTS_TABLE}
#     SELECT 
#         APRA_WORK_ID,
#         UPPER(APRA_TITLE) AS APRA_TITLE,
#         UPPER(APRA_TITLE_VARIANT) AS APRA_TITLE_VARIANT,
#         MUZOOKA_TRACK_ID,
#         UPPER(MUZOOKA_TITLE) AS MUZOOKA_TITLE,
#         UPPER(MUZOOKA_TITLE_VARIANT) AS MUZOOKA_TITLE_VARIANT,
#         TITLE_VARIANT_MATCH_SCORE,
#         UPPER(APRA_ISWC) AS APRA_ISWC,
#         UPPER(MUZOOKA_ISWC) AS MUZOOKA_ISWC,
#         PARTITION_ID,
#         UPPER(MATCH_TYPE) AS MATCH_TYPE
#     FROM {iswc_matches_table}
#     WHERE TITLE_VARIANT_MATCH_SCORE >= 0.8
#     """).collect()
    
#     # STEP 4: IDENTIFY RECORDS WITHOUT ISWC MATCHES
#     print("Step 4: Identifying records without ISWC matches...")
    
#     # Create a table of records that don't have matching ISWCs
#     session.sql(f"""
#     CREATE OR REPLACE TEMPORARY TABLE NO_ISWC_MATCH_{PARTITION_ID} AS
#     SELECT a.*
#     FROM ADC_VARIANTS_{PARTITION_ID} a
#     WHERE NOT EXISTS (
#         SELECT 1 
#         FROM ISWC_MATCHED_RECORDS_{PARTITION_ID} m
#         WHERE m.APRA_WORK_ID = a.APRA_WORK_ID
#     )
#     """).collect()
    
#     # Count records without ISWC matches
#     no_iswc_match_count = session.sql(f"SELECT COUNT(*) as count FROM NO_ISWC_MATCH_{PARTITION_ID}").collect()[0]['COUNT']
#     print(f"{no_iswc_match_count} records don't have ISWC matches, proceeding to title variant matching...")
    
#     # STEP 5: PERFORM TITLE VARIANT MATCHING FOR RECORDS WITHOUT ISWC MATCHES
#     if no_iswc_match_count > 0:
#         print("Step 5: Performing title variant matching for records without ISWC matches...")
        
#         # Calculate number of batches for title variant matching
#         total_batches = -(-no_iswc_match_count // BATCH_SIZE)  # Ceiling division
        
#         for batch_num in range(total_batches):
#             batch_start_time = time.time()
#             batch_start = batch_num * BATCH_SIZE
            
#             # Create and populate batch table
#             session.sql(f"""
#             CREATE OR REPLACE TEMPORARY TABLE VARIANT_BATCH_{PARTITION_ID} AS
#             SELECT *
#             FROM NO_ISWC_MATCH_{PARTITION_ID}
#             ORDER BY APRA_WORK_ID
#             LIMIT {BATCH_SIZE} OFFSET {batch_start}
#             """).collect()
            
#             # Count records in this batch
#             records_in_batch = session.sql(f"SELECT COUNT(*) as count FROM VARIANT_BATCH_{PARTITION_ID}").collect()[0]['COUNT']
            
#             # Skip if empty batch
#             if records_in_batch == 0:
#                 break
            
#             # Perform title variant matching only
#             variant_match_sql = f"""
#             INSERT INTO {variant_matches_table}
#             SELECT 
#                 a.APRA_WORK_ID,
#                 UPPER(a.TITLE) AS APRA_TITLE,  -- Extra UPPER to be absolutely sure
#                 UPPER(a.TITLE_VARIANT) AS APRA_TITLE_VARIANT,  -- Extra UPPER to be absolutely sure
#                 t.TRACK_ID AS MUZOOKA_TRACK_ID,
#                 UPPER(t.TITLE) AS MUZOOKA_TITLE,  -- Extra UPPER to be absolutely sure
#                 UPPER(t.TITLE_VARIANT) AS MUZOOKA_TITLE_VARIANT,  -- Extra UPPER to be absolutely sure
#                 GREATEST(
#                     -- Calculate exact match scores between title variants only
#                     CASE 
#                         WHEN a.UPPER_TITLE_VARIANT = UPPER(t.TITLE_VARIANT) THEN 1.0
#                         WHEN a.CLEAN_TITLE_VARIANT = UPPER(REGEXP_REPLACE(t.TITLE_VARIANT, '[^A-Z0-9\\\\s]', ' ')) THEN 0.95
#                         WHEN a.COMPACT_TITLE_VARIANT = UPPER(REGEXP_REPLACE(t.TITLE_VARIANT, '[^A-Z0-9]', '')) THEN 0.90
#                         WHEN a.NO_ARTICLE_TITLE_VARIANT = UPPER(REGEXP_REPLACE(REGEXP_REPLACE(t.TITLE_VARIANT, '[^A-Z0-9\\\\s]', ' '), '^(THE|A|AN)\\\\s+', '')) THEN 0.85
#                         ELSE TITLE_SIMILARITY(a.TITLE_VARIANT, t.TITLE_VARIANT)
#                     END
#                 ) AS TITLE_VARIANT_MATCH_SCORE,
#                 UPPER(a.ISWC) AS APRA_ISWC,  -- Extra UPPER to be absolutely sure
#                 UPPER(REGEXP_REPLACE(t.ISWC, '[^A-Za-z0-9]', '')) AS MUZOOKA_ISWC,
#                 {PARTITION_ID} AS PARTITION_ID,
#                 'VARIANT_MATCH' AS MATCH_TYPE
#             FROM 
#                 VARIANT_BATCH_{PARTITION_ID} a
#             CROSS JOIN 
#                 MZK_TRACKS_TITLE_VARIANTS t
#             WHERE
#                 t.TITLE_VARIANT IS NOT NULL AND TRIM(t.TITLE_VARIANT) != '' AND
#                 (
#                     -- Filter combinations for better performance - ONLY VARIANTS
#                     a.UPPER_TITLE_VARIANT = UPPER(t.TITLE_VARIANT) OR
#                     a.CLEAN_TITLE_VARIANT = UPPER(REGEXP_REPLACE(t.TITLE_VARIANT, '[^A-Z0-9\\\\s]', ' ')) OR
#                     a.COMPACT_TITLE_VARIANT = UPPER(REGEXP_REPLACE(t.TITLE_VARIANT, '[^A-Z0-9]', '')) OR
#                     a.NO_ARTICLE_TITLE_VARIANT = UPPER(REGEXP_REPLACE(REGEXP_REPLACE(t.TITLE_VARIANT, '[^A-Z0-9\\\\s]', ' '), '^(THE|A|AN)\\\\s+', '')) OR
#                     TITLE_SIMILARITY(a.TITLE_VARIANT, t.TITLE_VARIANT) >= 0.8
#                 )
#             """
#             session.sql(variant_match_sql).collect()
            
#             # Report batch progress
#             batch_variant_matches = session.sql(f"""
#                 SELECT COUNT(*) as count FROM {variant_matches_table}
#                 WHERE APRA_WORK_ID IN (SELECT APRA_WORK_ID FROM VARIANT_BATCH_{PARTITION_ID})
#             """).collect()[0]['COUNT']
            
#             batch_time = time.time() - batch_start_time
#             print(f"Title variant matching batch {batch_num + 1}/{total_batches}: {records_in_batch} records processed in {batch_time:.2f} seconds")
#             print(f"Found {batch_variant_matches} title variant matches in this batch")
            
#             # Clean up temporary batch table
#             session.sql(f"DROP TABLE IF EXISTS VARIANT_BATCH_{PARTITION_ID}").collect()
    
#         # STEP 6: INSERT TITLE VARIANT MATCHES INTO FINAL RESULTS TABLE
#         print("Step 6: Inserting title variant matches into final results table...")
        
#         # Insert with an additional UPPER() for each string field to double-ensure uppercase in the results
#         session.sql(f"""
#         INSERT INTO {RESULTS_TABLE}
#         SELECT 
#             APRA_WORK_ID,
#             UPPER(APRA_TITLE) AS APRA_TITLE,
#             UPPER(APRA_TITLE_VARIANT) AS APRA_TITLE_VARIANT,
#             MUZOOKA_TRACK_ID,
#             UPPER(MUZOOKA_TITLE) AS MUZOOKA_TITLE,
#             UPPER(MUZOOKA_TITLE_VARIANT) AS MUZOOKA_TITLE_VARIANT,
#             TITLE_VARIANT_MATCH_SCORE,
#             UPPER(APRA_ISWC) AS APRA_ISWC,
#             UPPER(MUZOOKA_ISWC) AS MUZOOKA_ISWC,
#             PARTITION_ID,
#             UPPER(MATCH_TYPE) AS MATCH_TYPE
#         FROM {variant_matches_table}
#         WHERE TITLE_VARIANT_MATCH_SCORE >= 0.8
#         """).collect()
    
#     # Verify that everything is now in uppercase
#     print("Checking sample of results to verify uppercase...")
#     sample_results = session.sql(f"""
#     SELECT MUZOOKA_TITLE, MUZOOKA_TITLE_VARIANT 
#     FROM {RESULTS_TABLE} 
#     WHERE PARTITION_ID = {PARTITION_ID}
#     LIMIT 5
#     """).collect()
    
#     for row in sample_results:
#         print(f"MUZOOKA_TITLE: {row['MUZOOKA_TITLE']}, MUZOOKA_TITLE_VARIANT: {row['MUZOOKA_TITLE_VARIANT']}")
    
#     # If there are any non-uppercase values, force them
#     non_upper_count = session.sql(f"""
#     SELECT COUNT(*) as count
#     FROM {RESULTS_TABLE}
#     WHERE 
#         PARTITION_ID = {PARTITION_ID} AND
#         (
#             MUZOOKA_TITLE != UPPER(MUZOOKA_TITLE) OR
#             MUZOOKA_TITLE_VARIANT != UPPER(MUZOOKA_TITLE_VARIANT) OR
#             APRA_TITLE != UPPER(APRA_TITLE) OR
#             APRA_TITLE_VARIANT != UPPER(APRA_TITLE_VARIANT)
#         )
#     """).collect()[0]['COUNT']
    
#     if non_upper_count > 0:
#         print(f"Found {non_upper_count} records with non-uppercase values. Forcing them to uppercase...")
#         session.sql(f"""
#         UPDATE {RESULTS_TABLE}
#         SET 
#             APRA_TITLE = UPPER(APRA_TITLE),
#             APRA_TITLE_VARIANT = UPPER(APRA_TITLE_VARIANT),
#             MUZOOKA_TITLE = UPPER(MUZOOKA_TITLE),
#             MUZOOKA_TITLE_VARIANT = UPPER(MUZOOKA_TITLE_VARIANT),
#             APRA_ISWC = UPPER(APRA_ISWC),
#             MUZOOKA_ISWC = UPPER(MUZOOKA_ISWC),
#             MATCH_TYPE = UPPER(MATCH_TYPE)
#         WHERE 
#             PARTITION_ID = {PARTITION_ID} AND
#             (
#                 MUZOOKA_TITLE != UPPER(MUZOOKA_TITLE) OR
#                 MUZOOKA_TITLE_VARIANT != UPPER(MUZOOKA_TITLE_VARIANT) OR
#                 APRA_TITLE != UPPER(APRA_TITLE) OR
#                 APRA_TITLE_VARIANT != UPPER(APRA_TITLE_VARIANT)
#             )
#         """).collect()
    
#     # Get final counts
#     iswc_matches = session.sql(f"""
#     SELECT COUNT(*) as count FROM {RESULTS_TABLE}
#     WHERE PARTITION_ID = {PARTITION_ID} AND MATCH_TYPE = 'ISWC_MATCH'
#     """).collect()[0]['COUNT']
    
#     variant_matches = session.sql(f"""
#     SELECT COUNT(*) as count FROM {RESULTS_TABLE}
#     WHERE PARTITION_ID = {PARTITION_ID} AND MATCH_TYPE = 'VARIANT_MATCH'
#     """).collect()[0]['COUNT']
    
#     total_time = time.time() - start_time
#     print(f"Partition {PARTITION_ID} complete!")
#     print(f"Processed {adc_count} records in {total_time:.2f} seconds")
#     print(f"Found {iswc_matches} ISWC matches and {variant_matches} title variant matches with score >= 0.8")
#     print(f"Total matches: {iswc_matches + variant_matches}")
#     print(f"All titles and variants are now in UPPERCASE in the results table")
    
#     # Clean up temporary objects
#     session.sql(f"DROP VIEW IF EXISTS ADC_VARIANTS_{PARTITION_ID}").collect()
#     session.sql(f"DROP TABLE IF EXISTS {iswc_matches_table}").collect()
#     session.sql(f"DROP TABLE IF EXISTS {variant_matches_table}").collect()
#     session.sql(f"DROP TABLE IF EXISTS ISWC_MATCHED_RECORDS_{PARTITION_ID}").collect()
#     session.sql(f"DROP TABLE IF EXISTS NO_ISWC_MATCH_{PARTITION_ID}").collect()
    
#     return adc_count, iswc_matches + variant_matches, total_time


# if __name__ == "__main__":
#     # Create Snowflake session
#     session = snowpark.Session.builder.getOrCreate()
    
#     # Create the similarity UDF if it doesn't exist yet
#     try:
#         session.sql("SELECT TITLE_SIMILARITY('test', 'test')").collect()
#         print("Title similarity UDF already exists.")
#     except Exception as e:
#         print("Creating title similarity UDF...")
#         title_similarity_js = """
#         CREATE OR REPLACE FUNCTION TITLE_SIMILARITY(str1 STRING, str2 STRING)
#         RETURNS FLOAT
#         LANGUAGE JAVASCRIPT
#         AS 
#         $$
#           // Normalize inputs - uppercase, remove extra spaces, special chars, etc.
#           function normalizeString(str) {
#             if (!str) return '';
#             return str.toUpperCase()
#                       .replace(/[^A-Z0-9\\s]/g, ' ')  // Replace special chars with space
#                       .replace(/\\s+/g, ' ')          // Replace multiple spaces with single space
#                       .trim();
#           }
          
#           // Jaro-Winkler similarity implementation
#           function jaroWinklerSimilarity(s1, s2) {
#             const normalized1 = normalizeString(s1);
#             const normalized2 = normalizeString(s2);
            
#             // If the strings are equal
#             if (normalized1 === normalized2) return 1.0;
            
#             // If either string is empty
#             if (normalized1.length === 0 || normalized2.length === 0) return 0.0;
            
#             // Maximum distance allowed for matching
#             const matchDistance = Math.floor(Math.max(normalized1.length, normalized2.length) / 2) - 1;
            
#             // Arrays to track matches
#             const s1Matches = Array(normalized1.length).fill(false);
#             const s2Matches = Array(normalized2.length).fill(false);
            
#             // Count of matches
#             let matches = 0;
            
#             // Look for matches
#             for (let i = 0; i < normalized1.length; i++) {
#               // Lower and upper bound for matching
#               const start = Math.max(0, i - matchDistance);
#               const end = Math.min(i + matchDistance + 1, normalized2.length);
              
#               for (let j = start; j < end; j++) {
#                 // Skip if already matched or not matching
#                 if (s2Matches[j] || normalized1[i] !== normalized2[j]) continue;
                
#                 // Found a match
#                 s1Matches[i] = true;
#                 s2Matches[j] = true;
#                 matches++;
#                 break;
#               }
#             }
            
#             // If no matches, return 0
#             if (matches === 0) return 0.0;
            
#             // Count transpositions
#             let transpositions = 0;
#             let k = 0;
            
#             for (let i = 0; i < normalized1.length; i++) {
#               if (!s1Matches[i]) continue;
              
#               while (!s2Matches[k]) k++;
              
#               if (normalized1[i] !== normalized2[k]) transpositions++;
              
#               k++;
#             }
            
#             // Calculate Jaro similarity
#             const jaroSim = (
#               (matches / normalized1.length) +
#               (matches / normalized2.length) +
#               ((matches - transpositions / 2) / matches)
#             ) / 3;
            
#             // Calculate prefix length (max 4)
#             let prefixLength = 0;
#             const maxPrefixLength = Math.min(4, Math.min(normalized1.length, normalized2.length));
            
#             for (let i = 0; i < maxPrefixLength; i++) {
#               if (normalized1[i] === normalized2[i]) {
#                 prefixLength++;
#               } else {
#                 break;
#               }
#             }
            
#             // Scaling factor for how much the score is adjusted by prefix length
#             const scalingFactor = 0.1;
            
#             // Calculate Jaro-Winkler similarity
#             return jaroSim + (prefixLength * scalingFactor * (1 - jaroSim));
#           }
          
#           // Main entry point for the UDF
#           return jaroWinklerSimilarity(STR1, STR2);
#         $$
#         """
#         session.sql(title_similarity_js).collect()
#         print("Title similarity UDF created successfully.")
    
#     # Process the partition
#     process_partition()


In [ ]:

#Match CROSS JOIN Tables



import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import col, lit, udf, call_udf, expr
import time
import pandas as pd
import traceback
import sys

def process_cross_join_title_variant_matching(input_table_num=0, mode="preview", limit=550000):
    
    # Define input and output table names based on the number
    input_table = f"ADC_MZK_VARIANTS_CROSS_JOIN_{input_table_num}"
    preview_results_table = f"TITLE_VARIANT_PREVIEW_RESULTS_{input_table_num}"
    complete_results_table = f"TITLE_VARIANT_COMPLETE_RESULTS_{input_table_num}"
    
    # Print header to make it clear which mode and table we're running
    if mode.lower() == "preview":
        print(f"RUNNING IN PREVIEW MODE FOR {input_table} - Processing only first {limit} records")
    else:
        print(f"RUNNING IN COMPLETE MODE FOR {input_table} - Processing entire dataset")
    
    # Create Snowflake session
    try:
        session = snowpark.Session.builder.getOrCreate()
    except Exception as e:
        print(f"ERROR: Failed to create Snowflake session: {e}")
        return 0, 0, 0
    
    # Variables to track progress even in case of failure
    total_processed = 0
    last_completed_batch = -1
    start_time = time.time()
    
    try:
        # Ensure the title similarity UDF exists
        try:
            session.sql("SELECT TITLE_SIMILARITY('test', 'test')").collect()
            print("Title similarity UDF already exists.")
        except Exception as e:
            print("Creating title similarity UDF...")
            create_jaro_winkler_similarity_udf(session)
        
        # Define results table name - use different tables for different modes
        results_table = preview_results_table if mode.lower() == "preview" else complete_results_table
        
        # Create a table to store the results with match scores - use CREATE IF NOT EXISTS to preserve existing results
        session.sql(f"""
        CREATE TABLE IF NOT EXISTS {results_table} (
            APRA_WORK_ID VARCHAR,
            APRA_TITLE_VARIANT VARCHAR,  -- Will store capitalized version
            MUZOOKA_TRACK_ID VARCHAR,    -- Will store capitalized version
            MUZOOKA_TITLE_VARIANT VARCHAR, -- Will store capitalized version
            TITLE_VARIANT_MATCH_SCORE FLOAT,
            APRA_ISWC VARCHAR,
            MUZOOKA_ISWC VARCHAR
        )
        """).collect()
        
        print(f"Creating enhanced view of the cross join with normalized and capitalized title variants for {input_table}...")
        
        # Create an enhanced view with normalized title variants and CAPITALIZED originals
        enhanced_view_sql = f"""
        CREATE OR REPLACE TEMPORARY VIEW ENHANCED_CROSS_JOIN AS
        SELECT 
            APRA_WORK_ID,
            UPPER(APRA_TITLE) AS APRA_TITLE_VARIANT,  -- Store capitalized version
            UPPER(MUZOOKA_TRACK_ID) AS MUZOOKA_TRACK_ID,  -- Capitalize track ID
            UPPER(MUZOOKA_TITLE) AS MUZOOKA_TITLE_VARIANT,  -- Store capitalized version
            -- Normalize APRA title variants (already uppercase from above)
            UPPER(APRA_TITLE) AS APRA_UPPER_TITLE_VARIANT,
            UPPER(REGEXP_REPLACE(APRA_TITLE, '[^A-Z0-9\\\\s]', ' ')) AS APRA_CLEAN_TITLE_VARIANT,
            UPPER(REGEXP_REPLACE(APRA_TITLE, '[^A-Z0-9]', '')) AS APRA_COMPACT_TITLE_VARIANT,
            UPPER(REGEXP_REPLACE(REGEXP_REPLACE(APRA_TITLE, '[^A-Z0-9\\\\s]', ' '), '^(THE|A|AN)\\\\s+', '')) AS APRA_NO_ARTICLE_TITLE_VARIANT,
            -- Normalize Muzooka title variants (already uppercase from above)
            UPPER(MUZOOKA_TITLE) AS MZK_UPPER_TITLE_VARIANT,
            UPPER(REGEXP_REPLACE(MUZOOKA_TITLE, '[^A-Z0-9\\\\s]', ' ')) AS MZK_CLEAN_TITLE_VARIANT,
            UPPER(REGEXP_REPLACE(MUZOOKA_TITLE, '[^A-Z0-9]', '')) AS MZK_COMPACT_TITLE_VARIANT,
            UPPER(REGEXP_REPLACE(REGEXP_REPLACE(MUZOOKA_TITLE, '[^A-Z0-9\\\\s]', ' '), '^(THE|A|AN)\\\\s+', '')) AS MZK_NO_ARTICLE_TITLE_VARIANT,
            APRA_ISWC,
            MUZOOKA_ISWC
        FROM 
            {input_table}
        """
        
        session.sql(enhanced_view_sql).collect()
        
        # Check for existing results and find the last completed batch
        try:
            # Get count of already processed records
            already_processed = session.sql(f"""
            SELECT COUNT(DISTINCT APRA_WORK_ID || '-' || MUZOOKA_TRACK_ID) as count
            FROM {results_table}
            """).collect()[0]['COUNT']
                
            if already_processed > 0:
                print(f"Found approximately {already_processed} already processed records")
                
                # Update total_processed to include already processed records
                total_processed = already_processed
        except Exception as e:
            print(f"Could not determine already processed records: {e}")
            # Continue with processing from the beginning
        
        # Get count of records to process
        if mode.lower() == "preview":
            # In preview mode, only count up to the limit
            record_count = min(
                session.sql("SELECT COUNT(*) as count FROM ENHANCED_CROSS_JOIN").collect()[0]['COUNT'],
                limit
            )
            print(f"Preview mode: Processing first {record_count} cross join records...")
        else:
            # In complete mode, process all records
            record_count = session.sql("SELECT COUNT(*) as count FROM ENHANCED_CROSS_JOIN").collect()[0]['COUNT']
            print(f"Complete mode: Processing all {record_count} cross join records...")
        
        # Define batch parameters
        BATCH_SIZE = 5000000
        total_batches = -(-record_count // BATCH_SIZE)  # Ceiling division
        
        # In preview mode, limit the number of batches
        if mode.lower() == "preview":
            total_batches = min(total_batches, -(-limit // BATCH_SIZE))
        
        # Start processing from the next batch after the last completed one
        start_batch = last_completed_batch + 1
        
        # Process each batch with error handling for individual batches
        for batch_num in range(start_batch, total_batches):
            batch_start_time = time.time()
            batch_start = batch_num * BATCH_SIZE
            
            try:
                # Calculate remaining rows to process in preview mode
                if mode.lower() == "preview" and (batch_start + BATCH_SIZE) > limit:
                    # Adjust batch size for the last batch in preview mode
                    current_batch_size = limit - batch_start
                else:
                    current_batch_size = BATCH_SIZE
                
                # Create batch - in preview mode, we need to limit the total records
                batch_sql = f"""
                CREATE OR REPLACE TEMPORARY TABLE BATCH_CROSS_JOIN AS
                SELECT *
                FROM ENHANCED_CROSS_JOIN
                LIMIT {current_batch_size} OFFSET {batch_start}
                """
                session.sql(batch_sql).collect()
                
                # Count records in batch
                records_in_batch = session.sql("SELECT COUNT(*) as count FROM BATCH_CROSS_JOIN").collect()[0]['COUNT']
                
                # Skip if empty batch
                if records_in_batch == 0:
                    print(f"Batch {batch_num} is empty, skipping")
                    break
                
                # Process matching for this batch - using same logic as before
                batch_match_sql = f"""
                INSERT INTO {results_table}
                SELECT 
                    APRA_WORK_ID,
                    APRA_TITLE_VARIANT,  -- Capitalized version
                    MUZOOKA_TRACK_ID,    -- Capitalized version
                    MUZOOKA_TITLE_VARIANT, -- Capitalized version
                    GREATEST(
                        -- Calculate exact match scores between title variants
                        CASE 
                            WHEN APRA_UPPER_TITLE_VARIANT = MZK_UPPER_TITLE_VARIANT THEN 1.0
                            WHEN APRA_CLEAN_TITLE_VARIANT = MZK_CLEAN_TITLE_VARIANT THEN 0.95
                            WHEN APRA_COMPACT_TITLE_VARIANT = MZK_COMPACT_TITLE_VARIANT THEN 0.90
                            WHEN APRA_NO_ARTICLE_TITLE_VARIANT = MZK_NO_ARTICLE_TITLE_VARIANT THEN 0.85
                            ELSE TITLE_SIMILARITY(APRA_TITLE_VARIANT, MUZOOKA_TITLE_VARIANT)
                        END
                    ) AS TITLE_VARIANT_MATCH_SCORE,
                    APRA_ISWC,
                    MUZOOKA_ISWC
                FROM 
                    BATCH_CROSS_JOIN
                WHERE
                    -- Filter by matching criteria - only keep good matches
                    APRA_UPPER_TITLE_VARIANT = MZK_UPPER_TITLE_VARIANT OR
                    APRA_CLEAN_TITLE_VARIANT = MZK_CLEAN_TITLE_VARIANT OR
                    APRA_COMPACT_TITLE_VARIANT = MZK_COMPACT_TITLE_VARIANT OR
                    APRA_NO_ARTICLE_TITLE_VARIANT = MZK_NO_ARTICLE_TITLE_VARIANT OR
                    TITLE_SIMILARITY(APRA_TITLE_VARIANT, MUZOOKA_TITLE_VARIANT) >= 0.8
                """
                session.sql(batch_match_sql).collect()
                
                # Count matches in this batch
                batch_matches = session.sql(f"""
                    SELECT COUNT(*) as count FROM {results_table}
                """).collect()[0]['COUNT'] - total_processed
                
                # Update total processed
                total_processed += records_in_batch
                
                batch_time = time.time() - batch_start_time
                percent_complete = (total_processed / record_count) * 100 if record_count > 0 else 100
                
                print(f"Progress: Batch {batch_num + 1}/{total_batches} - {total_processed}/{record_count} records ({percent_complete:.2f}%) ({batch_time:.2f} seconds)")
                #print(f"Found {batch_matches} new title variant matches in this batch")
                
                # Explicitly commit after each batch to ensure data is saved
                try:
                    session.sql("COMMIT").collect()
                    print(f"Successfully committed batch {batch_num}")
                except Exception as commit_error:
                    print(f"WARNING: Could not explicitly commit batch {batch_num}: {commit_error}")
                
                # If we're in preview mode and have reached the limit, stop processing
                if mode.lower() == "preview" and total_processed >= limit:
                    print(f"Preview limit of {limit} records reached. Stopping batch processing.")
                    break
                
            except Exception as batch_error:
                # Log the error but continue with the next batch
                print(f"ERROR processing batch {batch_num}: {batch_error}")
                print(f"Traceback: {traceback.format_exc()}")
                print(f"Continuing with next batch...")
                
                # Try to clean up the failed batch's temporary objects
                try:
                    session.sql("DROP TABLE IF EXISTS BATCH_CROSS_JOIN").collect()
                except:
                    pass
        
        # Update the original cross join table with match scores if in complete mode
        if mode.lower() == "complete":
            try:
                print(f"Updating original cross join table {input_table} with match scores...")
                
                # First, ensure the column exists
                update_sql = f"""
                ALTER TABLE {input_table} ADD COLUMN IF NOT EXISTS 
                TITLE_VARIANT_MATCH_SCORE FLOAT DEFAULT NULL
                """
                session.sql(update_sql).collect()
                
                # Update the scores
                update_sql = f"""
                MERGE INTO {input_table} target
                USING (
                    SELECT 
                        APRA_WORK_ID,
                        APRA_TITLE_VARIANT,
                        MUZOOKA_TRACK_ID,
                        MUZOOKA_TITLE_VARIANT,
                        TITLE_VARIANT_MATCH_SCORE
                    FROM {complete_results_table}
                ) source
                ON target.APRA_WORK_ID = source.APRA_WORK_ID 
                   AND UPPER(target.APRA_TITLE) = source.APRA_TITLE_VARIANT
                   AND UPPER(target.MUZOOKA_TRACK_ID) = source.MUZOOKA_TRACK_ID
                   AND UPPER(target.MUZOOKA_TITLE) = source.MUZOOKA_TITLE_VARIANT
                WHEN MATCHED THEN UPDATE SET
                    target.TITLE_VARIANT_MATCH_SCORE = source.TITLE_VARIANT_MATCH_SCORE
                """
                session.sql(update_sql).collect()
                print("Original cross join table updated successfully")
            except Exception as update_error:
                print(f"ERROR updating original cross join table: {update_error}")
                print(f"Traceback: {traceback.format_exc()}")
                print("Note: Results are still saved in the results table")
    
    except Exception as e:
        # Catch any unexpected errors in the main process
        print(f"ERROR in main processing: {e}")
        print(f"Traceback: {traceback.format_exc()}")
        print(f"Results are saved up to batch {last_completed_batch}")
    
    finally:
        # Always try to generate a report with whatever data was processed
        total_time = time.time() - start_time
        
        try:
            # Clean up temporary objects
            session.sql("DROP VIEW IF EXISTS ENHANCED_CROSS_JOIN").collect()
            session.sql("DROP TABLE IF EXISTS BATCH_CROSS_JOIN").collect()
            
            # Get statistics on what was successfully processed
            try:
                # Get match statistics
                match_stats = get_match_statistics(session, results_table)
                
                mode_text = "PREVIEW" if mode.lower() == "preview" else "COMPLETE"
                print(f"\n*** {mode_text} MODE RESULTS FOR {input_table} ***")
                print(f"Title variant matching {mode.lower()} mode completed!")
                print(f"Processed {total_processed} records in {total_time:.2f} seconds")
                print(f"Found {match_stats['TOTAL_MATCHES']} total title variant matches with score >= 0.8")
                print(f"Match types breakdown:")
                print(f"  - Exact matches (1.0): {match_stats['EXACT_MATCHES']}")
                print(f"  - Clean matches (0.95-0.99): {match_stats['CLEAN_MATCHES']}")
                print(f"  - Compact matches (0.90-0.94): {match_stats['COMPACT_MATCHES']}")
                print(f"  - No-article matches (0.85-0.89): {match_stats['NO_ARTICLE_MATCHES']}")
                print(f"  - Fuzzy matches (0.80-0.84): {match_stats['FUZZY_MATCHES']}")
                print(f"Score range: {match_stats['MIN_SCORE']:.2f} - {match_stats['MAX_SCORE']:.2f}, Average: {match_stats['AVG_SCORE']:.2f}")
                
                # Generate output report
                generate_output_report(session, results_table, mode_text, input_table_num)
                
                return total_processed, match_stats['TOTAL_MATCHES'], total_time
            except Exception as stats_error:
                print(f"ERROR generating statistics: {stats_error}")
                print(f"Traceback: {traceback.format_exc()}")
        except Exception as cleanup_error:
            print(f"ERROR during cleanup: {cleanup_error}")
        
        # Return what we know if we couldn't get statistics
        return total_processed, 0, total_time

def get_match_statistics(session, results_table):
    """
    Get statistics about match scores from the results table
    
    Args:
        session: Snowflake session
        results_table: Name of the results table
        
    Returns:
        Dictionary with match statistics
    """
    try:
        return session.sql(f"""
        SELECT 
            COUNT(*) AS total_matches,
            SUM(CASE WHEN TITLE_VARIANT_MATCH_SCORE = 1.0 THEN 1 ELSE 0 END) AS exact_matches,
            SUM(CASE WHEN TITLE_VARIANT_MATCH_SCORE >= 0.95 AND TITLE_VARIANT_MATCH_SCORE < 1.0 THEN 1 ELSE 0 END) AS clean_matches,
            SUM(CASE WHEN TITLE_VARIANT_MATCH_SCORE >= 0.9 AND TITLE_VARIANT_MATCH_SCORE < 0.95 THEN 1 ELSE 0 END) AS compact_matches,
            SUM(CASE WHEN TITLE_VARIANT_MATCH_SCORE >= 0.85 AND TITLE_VARIANT_MATCH_SCORE < 0.9 THEN 1 ELSE 0 END) AS no_article_matches,
            SUM(CASE WHEN TITLE_VARIANT_MATCH_SCORE >= 0.8 AND TITLE_VARIANT_MATCH_SCORE < 0.85 THEN 1 ELSE 0 END) AS fuzzy_matches,
            MIN(TITLE_VARIANT_MATCH_SCORE) AS min_score,
            MAX(TITLE_VARIANT_MATCH_SCORE) AS max_score,
            AVG(TITLE_VARIANT_MATCH_SCORE) AS avg_score
        FROM {results_table}
        """).collect()[0]
    except Exception as e:
        print(f"Error getting match statistics: {e}")
        # Return empty stats as fallback
        return {
            'TOTAL_MATCHES': 0, 
            'EXACT_MATCHES': 0,
            'CLEAN_MATCHES': 0,
            'COMPACT_MATCHES': 0,
            'NO_ARTICLE_MATCHES': 0,
            'FUZZY_MATCHES': 0,
            'MIN_SCORE': 0.0,
            'MAX_SCORE': 0.0,
            'AVG_SCORE': 0.0
        }

def create_jaro_winkler_similarity_udf(session):
    """
    Create JavaScript UDF for title similarity calculation using Jaro-Winkler distance
    
    Args:
        session: Snowflake session
        
    Returns:
        True if UDF was created successfully, False otherwise
    """
    try:
        print("Creating title similarity function with Jaro-Winkler algorithm...")
        
        # Create the JavaScript UDF using dollar-quoting to avoid escape issues
        title_similarity_js = """
        CREATE OR REPLACE FUNCTION TITLE_SIMILARITY(str1 STRING, str2 STRING)
        RETURNS FLOAT
        LANGUAGE JAVASCRIPT
        AS 
        $$
          // Normalize inputs - uppercase, remove extra spaces, special chars, etc.
          function normalizeString(str) {
            if (!str) return '';
            return str.toUpperCase()
                      .replace(/[^A-Z0-9\\s]/g, ' ')  // Replace special chars with space
                      .replace(/\\s+/g, ' ')          // Replace multiple spaces with single space
                      .trim();
          }
          
          // Jaro similarity implementation
          function jaroSimilarity(s1, s2) {
            // If the strings are equal
            if (s1 === s2) return 1.0;
            
            // If either string is empty
            if (s1.length === 0 || s2.length === 0) return 0.0;
            
            // Maximum distance allowed for matching
            const matchDistance = Math.floor(Math.max(s1.length, s2.length) / 2) - 1;
            
            // Arrays to track matches
            const s1Matches = Array(s1.length).fill(false);
            const s2Matches = Array(s2.length).fill(false);
            
            // Count of matches
            let matches = 0;
            
            // Look for matches
            for (let i = 0; i < s1.length; i++) {
              // Lower and upper bound for matching
              const start = Math.max(0, i - matchDistance);
              const end = Math.min(i + matchDistance + 1, s2.length);
              
              for (let j = start; j < end; j++) {
                // Skip if already matched or not matching
                if (s2Matches[j] || s1[i] !== s2[j]) continue;
                
                // Found a match
                s1Matches[i] = true;
                s2Matches[j] = true;
                matches++;
                break;
              }
            }
            
            // If no matches, return 0
            if (matches === 0) return 0.0;
            
            // Count transpositions
            let transpositions = 0;
            let k = 0;
            
            for (let i = 0; i < s1.length; i++) {
              if (!s1Matches[i]) continue;
              
              while (!s2Matches[k]) k++;
              
              if (s1[i] !== s2[k]) transpositions++;
              
              k++;
            }
            
            // Calculate Jaro similarity
            const jaroSim = (
              (matches / s1.length) +
              (matches / s2.length) +
              ((matches - transpositions / 2) / matches)
            ) / 3;
            
            return jaroSim;
          }
          
          // Jaro-Winkler similarity
          function jaroWinklerSimilarity(s1, s2) {
            const normalized1 = normalizeString(s1);
            const normalized2 = normalizeString(s2);
            
            // Calculate Jaro similarity
            const jaroSim = jaroSimilarity(normalized1, normalized2);
            
            // Calculate prefix length (max 4)
            let prefixLength = 0;
            const maxPrefixLength = Math.min(4, Math.min(normalized1.length, normalized2.length));
            
            for (let i = 0; i < maxPrefixLength; i++) {
              if (normalized1[i] === normalized2[i]) {
                prefixLength++;
              } else {
                break;
              }
            }
            
            // Scaling factor for how much the score is adjusted by prefix length
            const scalingFactor = 0.1;
            
            // Calculate Jaro-Winkler similarity
            return jaroSim + (prefixLength * scalingFactor * (1 - jaroSim));
          }
          
          // Main entry point for the UDF
          return jaroWinklerSimilarity(STR1, STR2);
        $$
        """
        
        # Execute the SQL statement
        session.sql(title_similarity_js).collect()
        print("Title similarity UDF with Jaro-Winkler distance created successfully.")
        return True
    except Exception as e:
        print(f"Error creating title similarity UDF: {e}")
        print(f"Traceback: {traceback.format_exc()}")
        return False

def resume_processing_from_checkpoint(input_table_num=0, mode="complete", limit=550000):
    """
    Resume processing from the last successful batch for a specific input table
    
    Args:
        input_table_num: Number suffix of the input table to process
        mode: "preview" or "complete" processing mode
        limit: Max records to process in preview mode
        
    Returns:
        tuple: (total_processed, matches_found, duration_seconds)
    """
    print(f"Resuming processing from last successful batch for table {input_table_num}...")
    return process_cross_join_title_variant_matching(input_table_num=input_table_num, mode=mode, limit=limit)

def process_multiple_tables(table_numbers, mode="preview", limit=550000):
    """
    Process multiple input tables in sequence
    
    Args:
        table_numbers: List of table number suffixes to process
        mode: "preview" or "complete" processing mode
        limit: Max records to process in preview mode per table
        
    Returns:
        dict: Results summary for each table processed
    """
    results = {}
    
    for table_num in table_numbers:
        print(f"\n{'='*80}")
        print(f"PROCESSING TABLE ADC_MZK_VARIANTS_CROSS_JOIN_{table_num}")
        print(f"{'='*80}\n")
        
        try:
            processed, matches, duration = process_cross_join_title_variant_matching(
                input_table_num=table_num,
                mode=mode,
                limit=limit
            )
            
            results[table_num] = {
                "processed": processed,
                "matches": matches,
                "duration_seconds": duration,
                "status": "success"
            }
            
        except Exception as e:
            print(f"ERROR processing table {table_num}: {e}")
            print(f"Traceback: {traceback.format_exc()}")
            
            results[table_num] = {
                "status": "error",
                "error": str(e)
            }
    
    # Print summary of all tables processed
    print("\n" + "="*80)
    print("PROCESSING SUMMARY FOR ALL TABLES")
    print("="*80)
    
    total_processed = 0
    total_matches = 0
    total_duration = 0
    
    for table_num, result in results.items():
        status = result["status"]
        if status == "success":
            processed = result["processed"]
            matches = result["matches"]
            duration = result["duration_seconds"]
            
            total_processed += processed
            total_matches += matches
            total_duration += duration
            
            print(f"Table {table_num}: Processed {processed} records, found {matches} matches in {duration:.2f} seconds - {status}")
        else:
            print(f"Table {table_num}: FAILED - {result.get('error', 'Unknown error')}")
    
    print(f"\nTOTAL: Processed {total_processed} records, found {total_matches} matches in {total_duration:.2f} seconds")
    
    return results

if __name__ == "__main__":
    try:
        # Set the default parameters
        mode = "complete"  # Change to "complete" for full processing
        limit = 550000    # Only used in preview mode
        
        # Process a single table - specify the table number here
        table_num = 1     # Change this to process a different table
        
        print(f"Processing table ADC_MZK_VARIANTS_CROSS_JOIN_{table_num}")
        processed, matches, duration = process_cross_join_title_variant_matching(
            input_table_num=table_num,
            mode=mode,
            limit=limit
        )
        
        print(f"Processing completed successfully!")
        print(f"Processed {processed} records, found {matches} matches in {duration:.2f} seconds")
        
        # Uncomment to process multiple tables
        # process_multiple_tables([0, 1, 2], mode=mode, limit=limit)
    
    except Exception as e:
        print(f"Critical error in main execution: {e}")
        print(f"Traceback: {traceback.format_exc()}")
        print("\nYou can resume processing from the last successful batch by running:")
        print("resume_processing_from_checkpoint(input_table_num=0, mode='complete")